In [1]:
# Low-RAM clone: run from the repo root (data/, runs/, experiments/ and the ember package are resolved from there).
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ember").is_dir() and _root != _root.parent:
    _root = _root.parent
assert (_root / "ember").is_dir(), "repo root (the folder with ember/) not found above the cwd"
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"cwd: {_root}")

cwd: /home/morg/NLP_2526b/oripetel/nlp_proj/nlp_proj


# EMBER §3.1–3.2, variable-width sparse MF — Harry Potter, whitened, variant `m1`

> **Concept:** Harry Potter · **model:** `google/gemma-2-2b-it` · **variant:** `m1` = `VWConfig(method="m1", refit=False)` · **seed:** 42
>
> Clone of `low_ram_demands/whitened.ipynb` (background-whitened + target-sharpened, low RAM) that keeps **only Steps 3.1
> (factorization) and 3.2 (feature identification)**. Step 3.3 (erasure), 3.4 and everything that needs the HF model were removed.
>
> **What changed vs. the source notebook**
> - 3.1.d fits the whitened matrix $A = X^\top$ with the shared variable-width loop `fit_varwidth(A, CFG, seed=SEED)` from
>   `experiments/varwidth/core.py` instead of `SparseMatrixFactorization` + WTA. The implementation and settings are the same as the
>   experiments in `results/varwidth/` (`python -m experiments.varwidth.run`), so this notebook reproduces their
>   `results/varwidth/Harry_Potter/<tag>/seed42/` runs (3.1.d prints the difference if that run exists).
> - 3.1.d′ fits the raw matrix $A_0 = E_{V^*}^\top$ with the **same** method and config (`fit_varwidth(A0, CFG, seed=SEED)`).
> - 3.2 runs the two-stage Gemini judge **only on the whitened fit**. For the original fit (3.2.e′) there are **no Gemini calls**: it
>   lists the $\rho \ge \tau$ candidates with their widths and top tokens, and those candidates stand in for `F_C0` in the
>   comparison cells (3.2.w).
>
> **Hyperparameters**
>
> | | value |
> |---|---|
> | sparse MF | `VWConfig(method="m1", refit=False)`; all other `VWConfig` fields at their defaults: rank 100, sparsity 0.01 (unless set above), floor 0.002, cap 0.03, ridge 1e-4, max_iter 20000, tol 1e-4, patience 200, churn_tol 0.005, init randn, m3_every 20, m3_window 0.02, m3_c 2.0 |
> | whitening | BACKGROUND `"wiki"`, TARGET `"page"`, GAMMA 1.0, ETA / EPS_REL from `experiments/whitening/data/gemma/defaults.json` (0.05 / 1e-4) — as `experiments/varwidth/prep.py` (the source notebook had GAMMA = 0) |
> | selection | TAU 2.0, CONFIDENCE 0.85, TOP_K_TOKENS 20, GEMINI_MODEL `models/gemini-3.5-flash-lite`, judge input = bare tokens |
>
> **Runs.** Caches go to `RUN_DIR = runs/variable_width/Harry_Potter/m1/` (`fit.pt`, `fit0.pt`, `judge.json`, `summary.json`);
> the extended sentences of 3.2.g to `runs/variable_width/Harry_Potter/ext_sentences.json` (shared by the five variants).

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026).

| Step | Paper section | What happens | Code |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $X^\top \approx ZY$ with sparse $Y$ (variable width per feature) | `experiments/varwidth/core.py` `fit_varwidth` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |

**Notation.** `Z` $\in \mathbb R^{d\times k}$ and `Y` $\in \mathbb R^{k\times|V^*|}$ use the paper's shapes. The **width** of feature $i$ is
$|\{t : Y_{i,t} \neq 0\}|$, the number of tokens it uses.

**Requirements:** `HF_TOKEN` if the checkpoint is gated, and a Gemini API key for the judge on the whitened fit (3.2.d–e, 3.2.g neighbours).


---
## 0. Setup

### 0.1 Hyperparameters

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 (Gemma-2-2B-it) | App. A.1 |
| budget | $K = \lceil s\,|V^*| \rceil \cdot k$ nonzeros in $Y$, $s$ = `CFG.sparsity` (0.01, or 0.02 for `fixed2`) | App. A.1 / `VWConfig` |
| width bounds | $m_{min} = \lceil 0.002\,|V^*|\rceil$, $m_{max} = \lceil 0.03\,|V^*|\rceil$ (pooling methods and m3; `fixed` ignores them) | `VWConfig.floor`, `.cap` |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 200 iters (rel. error and support churn < 0.005) / $10^{-4}$ | `VWConfig` |
| refit | off: $Y = Y^{dense}\odot S$ (as the paper) | `VWConfig.refit` |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |


In [2]:
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = "Harry Potter"
MODEL_NAME = "google/gemma-2-2b-it"     # or "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42

# ==== VARIANT: variable-width sparse MF (experiments/varwidth/, same code and settings as results/varwidth/) ====
import dataclasses

from experiments.varwidth.common import VWConfig, budgets
from experiments.varwidth.core import fit_varwidth

VARIANT = "m1"
VARIANT_CONFIGS = {
    "fixed1": VWConfig(method="fixed", refit=False),                 # the paper: WTA, every feature keeps ceil(0.01 n) tokens
    "fixed2": VWConfig(method="fixed", refit=False, sparsity=0.02),  # WTA with 2%: every feature keeps ceil(0.02 n) tokens
    "m1": VWConfig(method="m1", refit=False),                        # contribution pooling: global top-K of |y_ij|*||z_i||, floor/cap
    "m3": VWConfig(method="m3", refit=False),                        # entropy-allocated widths (exp H of the noise-cut top 2% of |y_i|)
    "rn": VWConfig(method="rn", refit=False),                        # pooling on |y_ij| / ||y_i||, floor/cap
}
CFG = VARIANT_CONFIGS[VARIANT]   # m1: contribution pooling: global top-K of |y_ij| * ||z_i|| (K = ceil(0.01 n) * k), each row within [floor, cap]
SEED = 42
# all other VWConfig fields stay at their defaults (rank 100, floor 0.002, cap 0.03, ridge 1e-4, max_iter 20000, tol 1e-4,
# patience 200, churn_tol 0.005, init randn, m3_every 20, m3_window 0.02, m3_c 2.0)
# ===================================================================================================================

# ---- Step 3.1: names the cells below use, taken from CFG ----------------
RANK = CFG.rank          # k
SPARSITY = CFG.sparsity  # s: total budget K = ceil(s * |V*|) * k
RIDGE = CFG.ridge        # lambda
MAX_ITER = CFG.max_iter  # T
PATIENCE = CFG.patience

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite
JUDGE_FORMAT = "tokens"   # Stage-1 judge input: "tokens" (bare tokens, paper; this notebook's original format)
                          # | "words" (token → the words it was cut from in S_C ∪ S_N, alphabetical)

# ---- background whitening + target sharpening (Instructions.txt, experiments/whitening/sharpen.py) ----
import re
import sys

sys.path.insert(0, "experiments/whitening")
from sharpen import BackgroundWhitener, Sharpener, Transform, principal_angle_sines

BACKGROUND = "wiki"      # "wiki" (Wikipedia token types) or "full" (whole vocabulary)
TARGET = "page"          # "page" (all tokens of the concept's English Wikipedia page) or "page_minus_wiki"
GAMMA = 1.0              # sharpening strength, as experiments/varwidth/prep.py (the source notebook had 0)
MODEL_SLUG = "gemma"     # "gemma" | "llama": selects experiments/whitening/data/<MODEL_SLUG>
SMOKE = False            # True: quick pass (short fit, 3 judged features)
assert BACKGROUND in ("wiki", "full"), BACKGROUND
assert TARGET in ("page", "page_minus_wiki"), TARGET
assert GAMMA >= 0, GAMMA
assert MODEL_SLUG == ("llama" if "llama" in MODEL_NAME.lower() else "gemma"), (MODEL_SLUG, MODEL_NAME)

WH_DIR = Path("experiments/whitening/data") / MODEL_SLUG
_defaults_path = WH_DIR / "defaults.json"     # the file experiments/varwidth/prep.py reads (no built-in fallback)
defaults = json.loads(_defaults_path.read_text())
ETA = defaults["eta"]            # target shrinkage (0.05)
EPS_REL = defaults["eps_rel"]    # eps = EPS_REL * mean(background eigenvalues) (1e-4)
print(f"whitening defaults from {_defaults_path}: eta={ETA:g} eps_rel={EPS_REL:g}")

if SMOKE:
    CFG = dataclasses.replace(CFG, max_iter=50, patience=5)
    MAX_ITER, PATIENCE = CFG.max_iter, CFG.patience

# ---- run directories: resume caches ----
concept_slug = re.sub(r"\W+", "_", CONCEPT).strip("_")
_smoke = "_smoke" if SMOKE else ""
RUN_DIR = Path("runs/variable_width") / concept_slug / f"{VARIANT}{_smoke}"   # fit.pt, fit0.pt, judge.json, summary.json
RUN_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path, default):
    return json.loads(path.read_text()) if path.exists() else default


def save_json(path: Path, obj) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")   # write then rename, so a crash never leaves a half file
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    tmp.replace(path)


def invalidate(directory: Path, names) -> None:
    """Move stale cache files aside (<name>.stale) so they are recomputed."""
    for name in names:
        p = directory / name
        if p.exists():
            p.replace(p.with_name(p.name + ".stale"))
            print(f"  moved stale {p} -> {p.name}.stale")

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | variant={VARIANT} | seed={SEED} | config={CFG}")
print(f"background={BACKGROUND} | target={TARGET} | gamma={GAMMA:g} | eta={ETA:g} | eps_rel={EPS_REL:g} | "
      f"smoke={SMOKE} | run dir={RUN_DIR}")

whitening defaults from experiments/whitening/data/gemma/defaults.json: eta=0.05 eps_rel=0.0001
concept='Harry Potter' | model=google/gemma-2-2b-it | variant=m1 | seed=42 | config=VWConfig(method='m1', rank=100, sparsity=0.01, floor=0.002, cap=0.05, ridge=0.0001, max_iter=20000, tol=0.0001, patience=200, churn_tol=0.005, init='randn', refit=False, floorcap=True, continuation=False, cont_start=0.05, cont_iters=2000, m3_every=20, m3_window=0.02, m3_c=2.0, m4_every=10)
background=wiki | target=page | gamma=1 | eta=0.05 | eps_rel=0.0001 | smoke=False | run dir=runs/variable_width/Harry_Potter/m1


/home/morg/NLP_2526b/oripetel/nlp_proj/nlp_proj/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [3]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

Gemini key found.


### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [4]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset
from ember.timing import Timer
from ember.utils import (
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    get_special_token_ids,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"   # the experiments (experiments/varwidth/run.py) also fit on CPU
print(f"device: {DEVICE}")

device: cpu


---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(sparse coefficients)}$$

In the paper a hard winner-takes-all (WTA) constraint on the rows of $Y$ makes each feature $\mathbf z_i$ use only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept. **In this notebook** the support of $Y$ is
chosen by the variant's allocation rule (`CFG.method`, see 3.1.d): the total number of nonzeros is fixed at
$K = \lceil s|V^*|\rceil\cdot k$, but except for `fixed` the **width** (tokens per feature) can differ between features.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. **Note:** for Gemma, TL folds the $\sqrt{d}$ embedding
multiplier into `W_E`, so the factorization runs in this *scaled* space.

In [5]:
# Low-RAM: steps 3.1-3.2 use only W_E and the tokenizer, so no TransformerLens model is built.
# W_E is read straight from the checkpoint and matches TL's: Gemma = HF embedding * sqrt(d_model), Llama = as stored;
# TL skips writing-weight centering for RMSNorm models. E_DTYPE is the dtype this notebook's TL model used.
import json
import math
from types import SimpleNamespace

from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError
from safetensors import safe_open

E_DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:  # Llama-3: TL also pads with eos
    tokenizer.pad_token = tokenizer.eos_token


def load_input_embeddings(model_name: str) -> torch.Tensor:
    """Reads only model.embed_tokens.weight from the checkpoint (no model is built)."""
    try:
        index = json.loads(Path(hf_hub_download(model_name, "model.safetensors.index.json")).read_text())
        shard = index["weight_map"]["model.embed_tokens.weight"]
    except EntryNotFoundError:  # single-file checkpoint
        shard = "model.safetensors"
    with safe_open(hf_hub_download(model_name, shard), framework="pt") as f:
        return f.get_tensor("model.embed_tokens.weight")


class TokenizerOnly:
    """The parts of a HookedTransformer that 3.1-3.2 use: tokenizer, to_tokens, to_str_tokens."""

    def __init__(self, tok):
        self.tokenizer = tok

    def to_tokens(self, prompts, padding_side="left"):
        self.tokenizer.padding_side = padding_side
        return self.tokenizer(prompts, return_tensors="pt", padding=True)["input_ids"]  # BOS added by the tokenizer

    def to_str_tokens(self, ids):
        return self.tokenizer.batch_decode(torch.as_tensor(ids).reshape(-1, 1), clean_up_tokenization_spaces=False)


tl_model = TokenizerOnly(tokenizer)  # keeps the old names, so the cells below run unchanged
act_generator = SimpleNamespace(model=tl_model, data_device="cpu")

E = load_input_embeddings(MODEL_NAME).to(E_DTYPE)  # (|V|, d)
if "gemma" in MODEL_NAME.lower():
    E.mul_(torch.tensor(math.sqrt(E.shape[1]), dtype=E_DTYPE))  # in place: no second (|V|, d) copy
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

E: (256000, 2304)  dtype=torch.float32


### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [6]:
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

Extracting token IDs: 100%|██████████| 120/120 [00:00<00:00, 943.14it/s]

|S_C| = 300   |S_N| = 300
token occurrences = 18611  ->  |V*| = 4780 (1.87% of the vocabulary)


### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [7]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
K_BUDGET, M_MIN, M_MAX = budgets(len(vstar_ids), CFG)
if CFG.method == "fixed":
    print(f"{CFG.method}: every feature keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens (K = {K_BUDGET})")
else:
    print(f"{CFG.method}: K = ceil(s*|V*|)*k = {K_BUDGET} nonzeros in total, widths in [{M_MIN}, {M_MAX}] "
          f"(mean {K_BUDGET / RANK:.1f} per feature)")

A: (2304, 4780)  (d x |V*|)
m1: K = ceil(s*|V*|)*k = 4800 nonzeros in total, widths in [10, 239] (mean 48.0 per feature)


### 3.1.c′ Background whitening and target sharpening

Instructions steps 1–4, in the row convention of the notebook (rows are tokens; `sharpen.py`):

$$U = (E_{V^*} - \mathbf 1\boldsymbol\mu_B^\top)\,W,\quad W = Q(\Lambda+\epsilon I)^{-1/2}Q^\top;\qquad
S_\eta = (1-\eta)\,\tfrac1m U_C^\top U_C + \eta I = P\,\mathrm{diag}(g)\,P^\top;\qquad X = U\,S_\eta^{\gamma/2}$$

1. **Background** ($\boldsymbol\mu_B$, $Q$, $\Lambda$ of $\Sigma_B$) is loaded from `WH_DIR/bg_<BACKGROUND>_stats.pt`. It was computed once per
   model, in the same space as `E` (TL `W_E`: Gemma rows already × $\sqrt d$; Llama unscaled). $\epsilon$ = `EPS_REL` × mean($\Lambda$).
   Check: $W\Sigma_B W = Q\,\mathrm{diag}\big(\lambda/(\lambda+\epsilon)\big)Q^\top \approx I$, except along the smallest
   eigenvalues, which $\epsilon$ deliberately damps.
2. **Whitening** centres on $\boldsymbol\mu_B$ (not on the mean of $V^*$), so the concept's shift away from the background stays in the data.
3. **Target**: the token ids of the concept's Wikipedia page (`WH_DIR/targets/<concept>.json`, key `TARGET`), whitened with the
   same $W$. The second moment is taken around 0, so it includes that shift. $g_j$ are **enrichment ratios** ($g_j > 1$: the target
   varies more than the background along $\mathbf p_j$); $\eta$ = `ETA` puts a floor under the directions the target does not use.
4. **Sharpening** with $S_\eta^{\gamma/2}$ applies one fixed map to every row of $V^*$, concept and neutral alike. `GAMMA = 0` skips it.

The combined map is `T` (`Transform`): `T.forward(rows)` $= (\text{rows}-\boldsymbol\mu_B)\,M$ with $M$ = `T.matrix` $= W S_\eta^{\gamma/2}$, and
`T.unmap(z)` $= z\,S_\eta^{-\gamma/2}W^{-1}$ maps directions back. `transform(rows)` is the float32 helper the analysis cells use.

**Diagnostics.** The largest enrichment ratios, and the contrastive-PCA directions $\mathbf p_1,\dots,\mathbf p_5$: the whitened
Wikipedia background token types (`bg_wiki_types.json`) projected on each, top 15 shown (`*` = in the target set). If they are
concept tokens, the signal is there before any factorization. With `GAMMA = 0` the same diagnostics use a $\gamma=1$ sharpener that is
not applied.

From here on `A` is $X^\top$; `E_vstar_orig` (= `E_vstar`) keeps the raw rows and `A0` the raw data matrix.

**Same as the experiments.** This cell is the code path of `experiments/varwidth/prep.py` (`whitening_transform` +
`prepare(matrix="whitened")`): background `bg_wiki_stats.pt` in float64, `BackgroundWhitener(stats, EPS_REL)`, target ids =
`targets/Harry_Potter.json["page"]` within the vocabulary minus special ids, `Sharpener(U_target, ETA, GAMMA)` with GAMMA = 1,
and `A = T.forward(E_vstar).float().T`. `A0 = E_vstar.T` is prep's `"plain"` matrix.

In [8]:
# Instructions steps 1-4: background whitening W, target second moment S_eta, sharpening S_eta^(gamma/2). float64, CPU.
E_vstar_orig = E_vstar                                    # raw rows (Gemma: TL-scaled), for the original fit 3.1.d'
A0 = E_vstar.T.contiguous().to(DEVICE)                    # (d, |V*|) raw

# ---- step 1: background statistics (built once per model by experiments/whitening) ----
BG_PATH = WH_DIR / f"bg_{BACKGROUND}_stats.pt"
TARGETS_PATH = WH_DIR / "targets" / f"{concept_slug}.json"
for _p, _what in [(BG_PATH, f"background statistics ({BACKGROUND})"), (TARGETS_PATH, f"target token ids for {CONCEPT!r}")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} is missing: the {_what} must be built first (experiments/whitening)")
stats_bg = torch.load(BG_PATH, map_location="cpu")
missing_keys = {"mu", "lam", "Q"} - set(stats_bg)
assert not missing_keys, f"{BG_PATH} lacks {missing_keys}"
assert stats_bg["Q"].shape == (D_MODEL, D_MODEL), f"{BG_PATH}: Q is {tuple(stats_bg['Q'].shape)}, expected d = {D_MODEL}"
stats_bg = {k: (v.double() if torch.is_tensor(v) else v) for k, v in stats_bg.items()}
bw = BackgroundWhitener(stats_bg, EPS_REL)

lam_bg = stats_bg["lam"].clamp_min(0)
lam_desc = lam_bg.sort(descending=True).values
shrink = lam_desc / (lam_desc + bw.eps)                   # eigenvalues of W Sigma_B W
n95 = math.ceil(0.95 * len(lam_desc))
print(f"background {BACKGROUND}: n = {stats_bg.get('n', '?')} types, d = {D_MODEL}, eps = {EPS_REL:g} x mean(lam) = {bw.eps:.3e}")
print(f"W Sigma_B W ~ I: max |lam/(lam+eps) - 1| over the top 95% of directions = {(shrink[:n95] - 1).abs().max():.3e}, "
      f"overall = {(shrink - 1).abs().max():.3e} (the smallest eigenvalues are damped by eps on purpose)")
print(f"cond(Sigma_B) = {(lam_desc[0] / lam_desc[-1]).item() if lam_desc[-1] > 0 else float('inf'):.3e}   "
      f"cond(Sigma_B + eps I) = {((lam_desc[0] + bw.eps) / (lam_desc[-1] + bw.eps)).item():.3e}")

# ---- step 3: target rows, whitened ----
targets = json.loads(TARGETS_PATH.read_text())
assert TARGET in targets, f"{TARGETS_PATH} has keys {list(targets)}, not {TARGET!r}"
target_ids = sorted({int(t) for t in targets[TARGET] if 0 <= int(t) < VOCAB_SIZE} - set(special))
assert target_ids, f"{TARGETS_PATH}[{TARGET!r}] has no usable token ids"
E_target = E.index_select(0, torch.tensor(target_ids, device=E.device)).detach().double().cpu()   # (m, d), same space as E_vstar
U_target = bw.whiten(E_target)                                                                     # (m, d)
target_set, vstar_set = set(target_ids), set(vstar_ids)
_codes31 = build_token_label_codes(CONCEPT, dataset, tokenizer)
tc_ids = {t for t in vstar_ids if _codes31.get(t, 0) == 1}                                         # T_C (concept-only)
print(f"|target| = {len(target_ids)} ({TARGET})   in V*: {len(target_set & vstar_set)} / {len(vstar_ids)}   "
      f"concept-only T_C covered: {len(target_set & tc_ids)} / {len(tc_ids)}")

# ---- step 4: sharpening; T = whitening (+ sharpening) ----
sh = Sharpener(U_target, ETA, GAMMA) if GAMMA > 0 else None
sh_diag = sh if sh is not None else Sharpener(U_target, ETA, 1.0)   # gamma = 0: diagnostics only, never applied
T = Transform(bw, sh)


def transform(rows: torch.Tensor) -> torch.Tensor:
    """Rows (n, d) in the original (TL) space -> X-space: (rows - mu_B) W S_eta^(gamma/2), float32."""
    return T.forward(rows.detach().cpu()).float()


E_X = transform(E_vstar)                                  # (|V*|, d)
E_vstar_X = E_X                                           # all analysis cells use it
A = E_X.T.contiguous().to(DEVICE)                         # (d, |V*|): 3.1.d fits on this
m_tilde = U_target.mean(0)                                # whitened target mean (the whitened shift Delta)
m_X = m_tilde if sh is None else m_tilde @ sh.Sp          # its image in X-space (= mean of transform(E_target))

# ---- checks ----
I_d = torch.eye(D_MODEL, dtype=torch.float64)
T_inv = bw.W_inv if sh is None else sh.Sm @ bw.W_inv      # row form: z = z~ @ T_inv
print(f"inverse: max |T.matrix @ (Sm @ W_inv) - I| = {(T.matrix @ T_inv - I_d).abs().max():.2e}")
print(f"row norms: raw mean {E_vstar.norm(dim=1).mean():.3f}   X-space mean {E_X.norm(dim=1).mean():.3f}   "
      f"cond(S_eta) = {(sh_diag.g[0] / sh_diag.g[-1]).item():.3e}   cond(T.matrix) = {torch.linalg.cond(T.matrix).item():.3e}")
del I_d

# ---- diagnostics: enrichment ratios ----
g_top = sh_diag.g[:10].tolist()
print(f"\nenrichment ratios g (eta = {ETA:g}; directions the target does not use sit at g = eta): top 10 = {', '.join(f'{v:.2f}' for v in g_top)}")
print(f"directions with g > 1: {int((sh_diag.g > 1).sum())} / {D_MODEL}   g > 2: {int((sh_diag.g > 2).sum())}   "
      f"stretch range applied: g^(gamma/2) in [{sh_diag.g[-1].item() ** (GAMMA / 2):.3f}, {sh_diag.g[0].item() ** (GAMMA / 2):.3f}]")

# ---- diagnostics: contrastive PCA on the whitened Wikipedia background types ----
N_CPCA, N_CPCA_TOK = 5, 15
WIKI_TYPES_PATH = WH_DIR / "bg_wiki_types.json"
if WIKI_TYPES_PATH.exists():
    wiki_ids = [int(t) for t in json.loads(WIKI_TYPES_PATH.read_text()) if 0 <= int(t) < VOCAB_SIZE]
    P5 = sh_diag.P[:, :N_CPCA]                                         # (d, 5)
    signs = torch.sign(m_tilde @ P5)                                   # orient each p_j towards the target mean
    signs[signs == 0] = 1.0
    M5 = bw.W @ (P5 * signs)                                           # whiten + project in one (d, 5) map
    scores = torch.cat([(E.index_select(0, torch.tensor(wiki_ids[i:i + 20_000], device=E.device)).detach().double().cpu()
                         - bw.mu) @ M5 for i in range(0, len(wiki_ids), 20_000)])   # (N_wiki, 5)
    cpca = {}
    for j in range(N_CPCA):
        top = torch.argsort(scores[:, j], descending=True)[:N_CPCA_TOK].tolist()
        toks = tokenizer.convert_ids_to_tokens([wiki_ids[i] for i in top])
        cpca[f"p{j + 1} (g={sh_diag.g[j]:.2f})"] = [f"{t}{' *' if wiki_ids[i] in target_set else ''}  {scores[i, j]:.2f}"
                                                   for t, i in zip(toks, top)]
    hit = {k: sum(s.split("  ")[0].endswith(" *") for s in v) for k, v in cpca.items()}
    print(f"\ncPCA: top {N_CPCA_TOK} of {len(wiki_ids)} whitened Wikipedia types on p_1..p_{N_CPCA} "
          f"(* = in the target set; per direction: {hit})")
    cpca_df = pd.DataFrame(cpca)
    cpca_df.index += 1
    display(cpca_df)
    del scores
else:
    print(f"\n{WIKI_TYPES_PATH} not found: cPCA diagnostic skipped")

print(f"\nA (transformed): {tuple(A.shape)}   A0 (raw): {tuple(A0.shape)}")

background wiki: n = 107472 types, d = 2304, eps = 0.0001 x mean(lam) = 2.712e-04
W Sigma_B W ~ I: max |lam/(lam+eps) - 1| over the top 95% of directions = 3.525e-04, overall = 2.512e-02 (the smallest eigenvalues are damped by eps on purpose)
cond(Sigma_B) = 3.314e+04   cond(Sigma_B + eps I) = 3.231e+04
|target| = 3124 (page)   in V*: 2592 / 4780   concept-only T_C covered: 1832 / 1870
inverse: max |T.matrix @ (Sm @ W_inv) - I| = 2.50e-14
row norms: raw mean 81.424   X-space mean 61.443   cond(S_eta) = 2.394e+02   cond(T.matrix) = 3.805e+02

enrichment ratios g (eta = 0.05; directions the target does not use sit at g = eta): top 10 = 14.20, 9.84, 8.65, 7.86, 7.74, 7.12, 6.44, 6.18, 5.94, 5.67
directions with g > 1: 892 / 2304   g > 2: 367   stretch range applied: g^(gamma/2) in [0.244, 3.768]

experiments/whitening/data/gemma/bg_wiki_types.json not found: cPCA diagnostic skipped

A (transformed): (2304, 4780)   A0 (raw): (2304, 4780)


### 3.1.d Fit variable-width sparse MF (`fit_varwidth`, variant `m1`)

The shared loop of `experiments/varwidth/core.py`, one iteration ($\lambda$ = `CFG.ridge`):

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y^{dense} \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A,\qquad
S \leftarrow \text{Allocate}(Y^{dense}, Z),\qquad Y \leftarrow Y^{dense}\odot S$$

then the rows of $Y$ are scaled to unit norm (the scale goes into $Z$'s columns). Initialization is the paper's $\mathcal N(0,1)$ from a
generator seeded with `SEED`.

**Allocation rule of this variant.** **m1, contribution pooling.** The score of entry $(i,t)$ is its contribution $|Y^{dense}_{i,t}|\,\|\mathbf z_i\|$. One global pool keeps exactly $K = \lceil 0.01\,|V^*|\rceil\cdot k$ entries (the paper's total budget): each feature first keeps its top $m_{min} = \lceil 0.002\,|V^*|\rceil$, the rest go to the highest scores overall, skipping features that reached $m_{max} = \lceil 0.03\,|V^*|\rceil$. Widths therefore differ per feature.

**Refit is off** (`refit=False`): the kept entries keep their dense values, $Y = Y^{dense}\odot S$, as in the paper; there is no
least-squares refit on the support.

**Stopping:** after `patience` = 200 iterations in which the relative error did not improve by ≥ `tol` = $10^{-4}$ **and** the
support churn stayed below 0.005; otherwise at `max_iter` = 20,000. The final iterate is returned.

`A` is $X^\top$ from 3.1.c′. The fit is cached in `RUN_DIR/fit.pt` (the `fit_varwidth` dict, the schema of `results/varwidth/*/fit.pt`,
plus a `meta` key); a cache made with another configuration (or another $V^*$) is moved aside as `*.stale` with the judge and summary
caches. If the experiments' run of the same config exists, the cell prints how far this fit is from it.


In [9]:
# Resume: RUN_DIR/fit.pt is loaded when its configuration matches; otherwise it (and the caches built on it) are moved aside.
from experiments.varwidth.common import run_dir as vw_run_dir

FIT_PATH = RUN_DIR / "fit.pt"
fit_meta = {"model": MODEL_NAME, "matrix": "whitened", "vstar_ids": vstar_ids, "background": BACKGROUND, "target": TARGET,
            "gamma": GAMMA, "eta": ETA, "eps_rel": EPS_REL, "n_target": len(target_ids), "config": CFG.to_dict(), "seed": SEED}
fit = torch.load(FIT_PATH, weights_only=False) if FIT_PATH.exists() else None
if fit is not None and fit.get("meta") != fit_meta:
    print(f"{FIT_PATH} was made with another configuration: re-fitting")
    invalidate(RUN_DIR, ["fit.pt", "judge.json", "summary.json"])
    fit = None
if fit is not None:
    print(f"loaded {FIT_PATH}")
else:
    fit = fit_varwidth(A, CFG, seed=SEED, device=DEVICE)
    fit["meta"] = fit_meta
    torch.save(fit, FIT_PATH)

Z = fit["Z"].float().cpu()      # (d, k)      X-space features z~_i (columns)
Y = fit["Y"].float().cpu()      # (k, |V*|)   sparse coefficients, variable width per row
widths = (Y != 0).sum(1)        # (k,)        tokens per feature
ridge_used = CFG.ridge          # fit_varwidth has no ridge back-off; 3.2.g reuses lambda
t_fit = {"elapsed": fit["elapsed"]}
print(f"\nfitted in {t_fit['elapsed']:.1f}s ({fit['n_iter']} iterations, stop: {fit['stop_reason']}, "
      f"lock-in at iteration {fit['lockin_iter']})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}  rel_err={fit['rel_err']:.4f}")

# Same run in the experiments (python -m experiments.varwidth.run ...): results/varwidth/<concept>/<tag>/seed<N>/fit.pt
EXP_FIT_PATH = vw_run_dir(CONCEPT, CFG.tag(), SEED) / "fit.pt"
if EXP_FIT_PATH.exists() and not SMOKE:
    _exp = torch.load(EXP_FIT_PATH, weights_only=False)
    print(f"experiments' fit {EXP_FIT_PATH}: rel_err={_exp['rel_err']:.4f}  n_iter={_exp['n_iter']}  "
          f"max|dZ|={(Z - _exp['Z'].float()).abs().max():.2e}  max|dY|={(Y - _exp['Y'].float()).abs().max():.2e}  "
          f"same supports: {torch.equal(Y != 0, _exp['Y'] != 0)}")
    del _exp
else:
    print(f"(no experiments' fit at {EXP_FIT_PATH} to compare with)")


fitted in 45.1s (436 iterations, stop: converged, lock-in at iteration 39)  Z: (2304, 100)  Y: (100, 4780)  rel_err=0.8510
experiments' fit /home/morg/NLP_2526b/oripetel/nlp_proj/nlp_proj/results/varwidth/Harry_Potter/m1_refit-False/seed42/fit.pt: rel_err=0.8510  n_iter=436  max|dZ|=6.96e-05  max|dY|=3.28e-07  same supports: True


#### 3.1.d-w Per-feature widths

The width of feature $i$ is the number of tokens with $Y_{i,t}\neq 0$. For `fixed` all widths are equal; for the other variants
the total is $K$ and each width lies in $[m_{min}, m_{max}]$. The widths of the concept features $F_C$ are shown in 3.2.f, after selection.

In [10]:
w = widths.double()
print(f"[{VARIANT}] widths: min={int(widths.min())}  median={int(widths.median())}  mean={w.mean():.1f}  max={int(widths.max())}  "
      f"sum={int(widths.sum())} (K = {K_BUDGET})  bounds [{M_MIN}, {M_MAX}]" + ("  (fixed: bounds unused)" if CFG.method == "fixed" else ""))
print(f"features at the floor: {int((widths == M_MIN).sum())}   at the cap: {int((widths == M_MAX).sum())}   "
      f"quantiles: " + "  ".join(f"q{int(q * 100)}={w.quantile(q):.0f}" for q in (0.1, 0.25, 0.5, 0.75, 0.9)))
_order = torch.sort(widths, descending=True, stable=True).indices.tolist()
print("widest 10 :", ", ".join(f"{f}:{int(widths[f])}" for f in _order[:10]))
print("narrowest 10:", ", ".join(f"{f}:{int(widths[f])}" for f in _order[::-1][:10]))
if widths.min() < widths.max():
    print("\nhistogram of widths:")
    print(pd.cut(widths.numpy(), bins=min(15, int(widths.max() - widths.min()) + 1)).value_counts().sort_index().to_string())

[m1] widths: min=10  median=29  mean=48.0  max=239  sum=4800 (K = 4800)  bounds [10, 239]
features at the floor: 3   at the cap: 2   quantiles: q10=14  q25=22  q50=30  q75=48  q90=106
widest 10 : 56:239, 66:239, 33:221, 86:220, 97:191, 82:171, 72:164, 95:123, 49:121, 93:117
narrowest 10: 84:10, 36:10, 27:10, 71:11, 60:11, 39:11, 65:12, 47:12, 44:12, 29:12

histogram of widths:
(9.771, 25.267]       45
(25.267, 40.533]      22
(40.533, 55.8]        15
(55.8, 71.067]         1
(71.067, 86.333]       3
(86.333, 101.6]        2
(101.6, 116.867]       2
(116.867, 132.133]     3
(132.133, 147.4]       0
(147.4, 162.667]       0
(162.667, 177.933]     2
(177.933, 193.2]       1
(193.2, 208.467]       0
(208.467, 223.733]     2
(223.733, 239.0]       2


### 3.1.d′ Reference fit on the original (untransformed) matrix

The **same method and config** (`CFG`, variant `m1`) and seed, `fit_varwidth(A0, CFG, seed=SEED)`, on the raw $A_0 = E_{V^*}^\top$
(prep's `"plain"` matrix; the experiments' `<tag>_plain` runs). `Z0` and `Y0` are used only for comparison: 3.1.e/f (errors),
3.2.e′ (the raw fit's $\rho\ge\tau$ candidates, **no Gemini calls**) and 3.2.w (feature matching). Cached in `RUN_DIR/fit0.pt`.

In [11]:
FIT0_PATH = RUN_DIR / "fit0.pt"
fit0_meta = {"model": MODEL_NAME, "matrix": "plain", "vstar_ids": vstar_ids, "config": CFG.to_dict(), "seed": SEED}
fit0 = torch.load(FIT0_PATH, weights_only=False) if FIT0_PATH.exists() else None
if fit0 is not None and fit0.get("meta") != fit0_meta:
    print(f"{FIT0_PATH} was made with another configuration: re-fitting")
    invalidate(RUN_DIR, ["fit0.pt"])
    fit0 = None
if fit0 is not None:
    print(f"loaded {FIT0_PATH}")
else:
    fit0 = fit_varwidth(A0, CFG, seed=SEED, device=DEVICE)
    fit0["meta"] = fit0_meta
    torch.save(fit0, FIT0_PATH)

Z0 = fit0["Z"].float().cpu()    # (d, k)      raw-space features
Y0 = fit0["Y"].float().cpu()    # (k, |V*|)   sparse coefficients
widths0 = (Y0 != 0).sum(1)      # (k,)
t_fit0 = {"elapsed": fit0["elapsed"]}
print(f"\nfitted in {t_fit0['elapsed']:.1f}s ({fit0['n_iter']} iterations, stop: {fit0['stop_reason']})  "
      f"Z0: {tuple(Z0.shape)}  Y0: {tuple(Y0.shape)}  rel_err={fit0['rel_err']:.4f}")
print(f"widths0: min={int(widths0.min())}  median={int(widths0.median())}  max={int(widths0.max())}  sum={int(widths0.sum())}")

EXP_FIT0_PATH = vw_run_dir(CONCEPT, CFG.tag() + "_plain", SEED) / "fit.pt"
if EXP_FIT0_PATH.exists() and not SMOKE:
    _exp = torch.load(EXP_FIT0_PATH, weights_only=False)
    print(f"experiments' fit {EXP_FIT0_PATH}: rel_err={_exp['rel_err']:.4f}  max|dY0|={(Y0 - _exp['Y'].float()).abs().max():.2e}  "
          f"same supports: {torch.equal(Y0 != 0, _exp['Y'] != 0)}")
    del _exp


fitted in 32.1s (312 iterations, stop: converged)  Z0: (2304, 100)  Y0: (100, 4780)  rel_err=0.8985
widths0: min=10  median=15  max=239  sum=4800
experiments' fit /home/morg/NLP_2526b/oripetel/nlp_proj/nlp_proj/results/varwidth/Harry_Potter/m1_refit-False_plain/seed42/fit.pt: rel_err=0.8985  max|dY0|=4.92e-07  same supports: True


### 3.1.e Check the factorization

Reconstruction quality and sparsity. The features use $K$ tokens in total: exactly $\lceil s|V^*|\rceil$ each for `fixed`,
otherwise between $m_{min}$ and $m_{max}$. Tokens that no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

Both fits are reported, each against its own data matrix. The two errors are not comparable in absolute terms: whitening
flattens the spectrum, so the transformed error is higher for any rank-$k$ model (the SVD baseline in 3.1.f shows by how much).

The second cell is the optional check of Instructions step 6: $|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ between each fitted factor
and the target mean in X-space ($\tilde{\mathbf m} = $ `m_X`). Concept factors should be among the highest; low values flag generic factors.


In [12]:
A_cpu, A0_cpu = A.cpu(), A0.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()        # transformed fit, X-space
rel_err0 = (torch.norm(A0_cpu - Z0 @ Y0) / torch.norm(A0_cpu)).item()   # original fit, raw space
nz = Y.abs() > 0
nz0 = Y0.abs() > 0

for name, err, mask in [("transformed", rel_err, nz), ("original", rel_err0, nz0)]:
    tokens_per_feature, features_per_token = mask.sum(1), mask.sum(0)
    print(f"[{name}] relative reconstruction error ||A - ZY||_F / ||A||_F = {err:.4f}")
    print(f"[{name}] tokens per feature : min={tokens_per_feature.min().item()}  median={int(tokens_per_feature.median())}  "
          f"max={tokens_per_feature.max().item()}  total={int(tokens_per_feature.sum())}")
    print(f"[{name}] features per token : mean={features_per_token.float().mean():.2f}  "
          f"max={features_per_token.max().item()}  uncovered tokens={int((features_per_token == 0).sum())}\n")

[transformed] relative reconstruction error ||A - ZY||_F / ||A||_F = 0.8510
[transformed] tokens per feature : min=10  median=29  max=239  total=4800
[transformed] features per token : mean=1.00  max=10  uncovered tokens=2024

[original] relative reconstruction error ||A - ZY||_F / ||A||_F = 0.8984
[original] tokens per feature : min=10  median=15  max=239  total=4800
[original] features per token : mean=1.00  max=4  uncovered tokens=1156



In [13]:
# Instructions step 6 (optional check): |cos| of each X-space factor with the target mean m_X.
cos_m = F.cosine_similarity(Z.double(), m_X.unsqueeze(1), dim=0).abs()       # (k,)
cos_m_order = torch.argsort(cos_m, descending=True).tolist()
print(f"|cos(z~_i, m_X)|: median={cos_m.median():.3f}  max={cos_m.max():.3f}  "
      f"(random-direction scale ~ {math.sqrt(2 / (math.pi * D_MODEL)):.3f})")
cos_m_df = pd.DataFrame({
    "feature": cos_m_order[:10],
    "|cos(z~, m_X)|": cos_m[cos_m_order[:10]].numpy(),
    "top tokens": [", ".join(tokenizer.convert_ids_to_tokens(
        [vstar_ids[j] for j in torch.argsort(Y[f].abs(), descending=True)[:8].tolist()])) for f in cos_m_order[:10]],
})
cos_m_df

|cos(z~_i, m_X)|: median=0.160  max=0.560  (random-direction scale ~ 0.017)


,feature,"|cos(z~, m_X)|",top tokens
0,86,0.559852,"▁high, ▁great, ▁big, ▁major, ▁important, ▁strong, ▁large, ▁real"
1,66,0.558904,"▁question, ▁site, ▁issue, ▁area, ▁process, ▁line, ▁story, ▁section"
2,56,0.460858,"▁created, ▁presented, ▁made, ▁designed, ▁described, ▁produced, ▁found, ▁released"
3,15,0.456191,"▁a, ▁the, ▁an, ▁its, ▁their, ▁his, the, ▁another"
4,82,0.432721,"▁take, ▁remain, ▁stay, ▁go, ▁have, ▁serve, ▁receive, ▁continue"
5,97,0.418283,"▁topics, ▁themes, ▁stories, ▁items, ▁elements, ▁issues, ▁scenes, ▁aspects"
6,49,0.412397,"▁speaking, ▁coming, ▁saying, ▁being, ▁talking, ▁taking, ▁showing, ▁seeing"
7,95,0.411722,"▁within, ▁in, ▁during, ▁with, ▁among, ▁for, ▁at, ▁from"
8,48,0.387319,"▁were, ▁could, ▁was, ▁would, ▁might, ▁did, ▁may, ▁had"
9,64,0.353578,"▁this, ▁these, These, This, ▁those, ▁the, the, The"


In [14]:
print(Z.size())
print(Y.size())
print(RANK)

torch.Size([2304, 100])
torch.Size([100, 4780])
100


### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the sparsity of $Y$ costs in reconstruction.

In [15]:
for name, A_mat, err, mask, Z_mat in [("transformed", A_cpu, rel_err, nz, Z), ("original", A0_cpu, rel_err0, nz0, Z0)]:
    U, S, Vh = torch.linalg.svd(A_mat, full_matrices=False)
    A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
    svd_rel_err = (torch.norm(A_mat - A_svd) / torch.norm(A_mat)).item()

    svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
    smf_nnz = Z_mat.numel() + int(mask.sum())
    print(f"[{name}] rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
    print(f"[{name}] rank-{RANK} Sparse MF     : relative error = {err:.4f}   (nonzero parameters: {smf_nnz:,})")

    # Smallest SVD rank whose error is no worse than Sparse MF's.
    tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
    matching_rank = int((tail <= err).nonzero()[0]) if (tail <= err).any() else len(S)
    print(f"[{name}] SVD reaches the Sparse MF error ({err:.4f}) at rank {matching_rank}\n")
del U, S, Vh, A_svd

[transformed] rank-100 truncated SVD : relative error = 0.7917   (parameters: 708,500)
[transformed] rank-100 Sparse MF     : relative error = 0.8510   (nonzero parameters: 235,200)
[transformed] SVD reaches the Sparse MF error (0.8510) at rank 53

[original] rank-100 truncated SVD : relative error = 0.8219   (parameters: 708,500)
[original] rank-100 Sparse MF     : relative error = 0.8984   (nonzero parameters: 235,200)
[original] SVD reaches the Sparse MF error (0.8984) at rank 12



---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [16]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

T_C (concept-only) = 1870   T_N (neutral-only) = 2229   both = 681


### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [17]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

rho >= 1.0 :  81 / 100 features
rho >= 1.25:  78 / 100 features
rho >= 1.5 :  74 / 100 features
rho >= 1.75:  67 / 100 features
rho >= 2.0 :  63 / 100 features
rho >= 2.25:  57 / 100 features

63 candidate features pass tau = 2.0


,feature,ratio_abs,mean_abs_concept,mean_abs_neutral,num_concept,num_neutral,num_both
46,46,129.348945,0.003137,0.000024,44,1,3
88,88,53.965477,0.002714,0.000050,49,2,4
71,71,44.848994,0.001346,0.000030,9,1,1
44,44,34.419179,0.001133,0.000033,10,1,1
47,47,28.225249,0.000981,0.000035,9,1,2
...,...,...,...,...,...,...,...
1,1,2.063697,0.001414,0.000685,18,12,13
42,42,2.038758,0.001076,0.000528,9,7,1
67,67,2.035762,0.000838,0.000411,10,7,3
86,86,2.020338,0.003426,0.001696,102,65,53


### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

In [18]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
if SMOKE:
    cand = cand.head(3)  # smoke: judge only the 3 highest-rho candidates
from ember.judge_words import desc_sys_prompt, judge_cache_name, judge_items, token_word_counts

DESC_SYS = desc_sys_prompt(JUDGE_FORMAT, multilingual=False)   # = TOKENS_DESC_SYS for "tokens"
jw_counts = token_word_counts(tokenizer, [s for s, _ in dataset]) if JUDGE_FORMAT == "words" else {}


def judge_input(Ymat: torch.Tensor, f: int, activating_tokens) -> list:
    """What Stage 1 sees for feature f of Ymat: "tokens" = the top activating tokens (as before); "words" = token → source words."""
    if JUDGE_FORMAT == "tokens":
        return list(activating_tokens[:TOP_K_TOKENS])
    mags = Ymat[f].abs()
    top = torch.argsort(mags, descending=True)[:TOP_K_TOKENS]
    top = top[mags[top] > 0].tolist()
    return judge_items([vstar_ids[j] for j in top], lambda t: tokenizer.decode([t]), jw_counts, JUDGE_FORMAT,
                       multilingual=False)


cand["judge_tokens"] = [judge_input(Y, int(f), toks) for f, toks in zip(cand["feature"], cand["activating_tokens"])]
cand[["feature", "ratio_abs", "judge_tokens"]]

,feature,ratio_abs,judge_tokens
0,46,129.348945,"[ffindor, therin, umbledore, Dumbledore, Weasley, Malfoy, Hogwarts, Rowling, Potter, Hermione, Hermione, demort, Voldemort, Potter, Harry, Snape, Harry, magic, Magic, magic]"
1,88,53.965477,"[ magic, magic, Magic, Magic, magical, magical, Magical, wizard, wizards, witchcraft, Wizard, wizard, Wizards, witch, witches, Witch, Sorcerer, fantasy, fairytale, wiz]"
2,71,44.848994,"[ entry, enter, enters, entering, entries, into, foray, nter, Entertainment, infiltr, joining]"
3,44,34.419179,"[ read, reads, Read, reading, learn, literacy, listened, ride, write, learns, czy, rides]"
4,47,28.225249,"[ world, World, World, worlds, universe, worldwide, internationally, International, word, Universal, nation, earth]"
...,...,...,...
58,1,2.063697,"[ previous, previously, originally, original, earlier, last, initially, first, initial, Originally, formerly, former, old, First, Last, past, precedent, already, inaugural, new]"
59,42,2.038758,"[ white, white, White, White, black, Black, dark, Dark, darker, red, silver, Brown, green, purple, Whit, red, yellowish]"
60,67,2.035762,"[ award, Award, awards, Awards, awarded, Prizes, winner, medal, grant, nominated, confer, nominations, recognition, praise, contest, praised, bery, ceremony, championship, AW]"
61,86,2.020338,"[ high, great, big, major, important, strong, large, real, long, main, significant, full, good, primary, high, key, fundamental, principal, general, new]"


### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [19]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Persisted in RUN_DIR/judge.json, saved after every Gemini call, so a restart re-queries nothing.
# 3.1.d moves judge.json aside whenever it re-fits.
JUDGE_PATH = RUN_DIR / judge_cache_name(JUDGE_FORMAT, "tokens")   # judge.json for the original format
_judge = load_json(JUDGE_PATH, {})
descriptions = {int(f): d for f, d in _judge.get("descriptions", {}).items()}
verdicts = {int(f): tuple(v) for f, v in _judge.get("verdicts", {}).items()}


def save_judge() -> None:
    save_json(JUDGE_PATH, {"descriptions": descriptions, "verdicts": verdicts})

print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

126 requests needed, 0 descriptions and 0 verdicts cached; throttle=off


### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [20]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"], sys_prompt=DESC_SYS)
    save_judge()
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

---- Stage 1 prompt ----
You get tokens that represent a single feature vector (with some noise).
Infer the single most specific, cohesive concept shared by the relevant tokens.
Return only one concise sentence. No preface, no list, no caveats.

Example:
Tokens: ['▁tomorrow','▁tonight','▁yesterday','▁today','▁demain']
Explanation of feature behavior: this vector is related to specific dates and times (e.g., today/tomorrow/yesterday). 

Tokens: {tokens}
Explanation of feature behavior:


Stage 1: describe: 100%|██████████| 63/63 [00:39<00:00,  1.59it/s]


,feature,ratio_abs,description
0,46,129.348945,"This vector is related to characters, locations, and terminology from the Harry Potter universe."
1,88,53.965477,"This vector activates on words related to magic, sorcery, and fantasy folklore."
2,71,44.848994,"This feature vector represents the concept of entering or going into a space, state, or activity."
3,44,34.419179,"This vector activates for verbs and concepts related to acquiring knowledge, language skills, or literacy activities."
4,47,28.225249,"This feature vector represents the concept of the global earth, international scope, and the broader universe."
...,...,...,...
58,1,2.063697,"This vector represents concepts of prior occurrence, sequence, or historical precedence."
59,42,2.038758,This vector represents color descriptors and chromatic attributes.
60,67,2.035762,"This vector activates for terms related to honors, awards, and competitive recognition."
61,86,2.020338,"These tokens represent the concept of magnitude, importance, or scale."


In [21]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  width={int(widths[feat])}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 46  width=48  rho=129.35 ===
  Gemini : This vector is related to characters, locations, and terminology from the Harry Potter universe.
  tokens : 'ffindor'[C] 0.36, 'therin'[C] 0.30, 'umbledore'[C] 0.24, '▁Dumbledore'[C] 0.21, '▁Weasley'[C] 0.20, '▁Malfoy'[C] 0.19, '▁Hogwarts'[C] 0.19, '▁Rowling'[C] 0.19, '▁Potter'[C] 0.19, 'Hermione'[C] 0.18

=== feature 88  width=55  rho=53.97 ===
  Gemini : This vector activates on words related to magic, sorcery, and fantasy folklore.
  tokens : '▁magic'[C] 0.36, 'magic'[C] 0.35, 'Magic'[C] 0.34, '▁Magic'[B] 0.33, '▁magical'[C] 0.28, 'magical'[C] 0.26, '▁Magical'[C] 0.23, '▁wizard'[C] 0.18, '▁wizards'[C] 0.17, '▁witchcraft'[C] 0.15

=== feature 71  width=11  rho=44.85 ===
  Gemini : This feature vector represents the concept of entering or going into a space, state, or activity.
  tokens : '▁entry'[C] 0.47, '▁enter'[C] 0.45, '▁enters'[C] 0.43, '▁entering'[C] 0.43, '▁entries'[C] 0.40, '▁into'[B] 0.14, '▁foray'[C] 0.10, 'nter'[C] 0.09, 

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [22]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: descriptions[f] for f in cand["feature"].astype(int).tolist() if f in descriptions and f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)
    save_judge()

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

---- Stage 2 prompt ----
You get a concept name and a feature's description.
Decide if this feature describes the given concept. Consider if the feature is DISTINCTIVE for the concept (not broad like "sport" for "basketball") and not too noisy.
A feature should be marked true only if the concept is central and dominant in the description, not just one of several themes.
Return ONLY JSON: {"is_member": true|false, "confidence": 0..1}

Example 1:
Concept: Charity
Description: This vector represents elements related to giving and donations to charitable causes.
Answer: {"is_member": true, "confidence": 0.98}

Example 2:
Concept: Charity
Description: This vector represents elements related to money, such as gambling, investing and charity.
Answer: {"is_member": false, "confidence": 0.80}

Concept: Harry Potter
Description: {description}
Answer:


Stage 2: classify: 100%|██████████| 63/63 [00:37<00:00,  1.68it/s]


F_C = [46]  (1 concept-related features)


,feature,ratio_abs,is_member,confidence,selected,description
0,46,129.348945,True,1.00,True,"This vector is related to characters, locations, and terminology from the Harry Potter universe."
1,88,53.965477,False,0.90,False,"This vector activates on words related to magic, sorcery, and fantasy folklore."
2,71,44.848994,False,0.99,False,"This feature vector represents the concept of entering or going into a space, state, or activity."
3,44,34.419179,False,0.99,False,"This vector activates for verbs and concepts related to acquiring knowledge, language skills, or literacy activities."
4,47,28.225249,False,0.99,False,"This feature vector represents the concept of the global earth, international scope, and the broader universe."
...,...,...,...,...,...,...
58,1,2.063697,False,0.99,False,"This vector represents concepts of prior occurrence, sequence, or historical precedence."
59,42,2.038758,False,1.00,False,This vector represents color descriptors and chromatic attributes.
60,67,2.035762,False,0.99,False,"This vector activates for terms related to honors, awards, and competitive recognition."
61,86,2.020338,False,0.99,False,"These tokens represent the concept of magnitude, importance, or scale."


### 3.2.e′ Original fit: $\rho \ge \tau$ candidates (no Gemini)

The mass ratio ($\tau$ = 2.0) on `Y0`, **without the judge**: no Gemini calls are made for the original fit. Every candidate is
listed with its width and its top-15 tokens by $|Y^0_{i,t}|$ with labels [C]/[N]/[B], to compare by eye with the whitened $F_C$
(3.2.e/f). In the comparison cells below (3.2.w), `F_C0` **is this candidate list** ($\rho\ge\tau$ on the original fit), not a
judged selection. The last line lists $|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ for $i\in F_C$.

In [23]:
# tau filter on the original fit (Y0), no judge. F_C0 below = the rho >= tau candidates (NOT judged by Gemini).
N_SHOW0 = 15
stats0 = compute_embedding_stats(Y0.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)
for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"original fit: rho >= {thr:<4}: {(stats0['ratio_abs'] >= thr).sum():3d} / {RANK} features")
cand0 = stats0[stats0["ratio_abs"] >= TAU][["feature", "ratio_abs", "num_concept", "num_neutral", "num_both"]].reset_index(drop=True)
cand0["width"] = cand0["feature"].map(lambda f: int(widths0[int(f)]))
F_C0 = sorted(cand0["feature"].astype(int).tolist())   # stand-in for the judged F_C0: rho >= tau candidates of the original fit
print(f"\noriginal fit: {len(cand0)} candidates pass tau = {TAU}  ->  F_C0 (rho candidates, no judge) = {F_C0}   "
      f"(whitened fit, judged: F_C = {F_C})\n")

for _, row in cand0.iterrows():
    feat = int(row["feature"])
    mags = Y0[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW0]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
    print(f"=== original feature {feat}  width={int(widths0[feat])}  rho={row['ratio_abs']:.2f} ===")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

print("|cos(z~_i, m_X)| for i in F_C: " + (", ".join(
    f"{f}: {cos_m[f]:.3f} (rank {cos_m_order.index(f) + 1}/{RANK})" for f in F_C) or "F_C is empty"))
cand0

original fit: rho >= 1.0 :  68 / 100 features
original fit: rho >= 1.25:  62 / 100 features
original fit: rho >= 1.5 :  54 / 100 features
original fit: rho >= 1.75:  49 / 100 features
original fit: rho >= 2.0 :  48 / 100 features
original fit: rho >= 2.25:  47 / 100 features

original fit: 48 candidates pass tau = 2.0  ->  F_C0 (rho candidates, no judge) = [3, 5, 6, 10, 12, 18, 20, 22, 23, 24, 27, 28, 29, 30, 32, 35, 36, 39, 40, 41, 43, 50, 51, 54, 57, 58, 59, 61, 67, 70, 71, 73, 76, 78, 80, 81, 82, 83, 84, 85, 86, 87, 89, 91, 93, 94, 96, 98]   (whitened fit, judged: F_C = [46])

=== original feature 82  width=13  rho=117025.56 ===
  tokens : '▁book'[B] 0.41, '▁Book'[B] 0.39, 'book'[C] 0.39, '▁books'[C] 0.38, '▁Books'[C] 0.36, '▁novels'[C] 0.21, '▁novel'[B] 0.21, '▁bookstores'[C] 0.20, '▁Novel'[C] 0.19, '▁textbook'[C] 0.18, '▁ebooks'[C] 0.17, '▁novelist'[C] 0.12, '▁literature'[B] 0.11

=== original feature 57  width=23  rho=46.27 ===
  tokens : '▁magic'[C] 0.34, 'Magic'[C] 0.33, '▁Magi

,feature,ratio_abs,num_concept,num_neutral,num_both,width
0,82,117025.559302,9,0,4,13
1,57,46.272176,21,1,1,23
2,50,16.729407,7,1,2,10
3,18,12.558961,14,2,1,17
4,39,11.569518,11,1,3,15
5,40,9.087780,7,2,1,10
6,22,7.690376,8,2,1,11
7,23,7.132580,14,2,0,16
8,36,6.509302,6,3,1,10
9,98,6.266172,9,1,0,10


In [24]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  width={int(widths[feat])}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 46  width=48  rho=129.35 ===
  Gemini  : This vector is related to characters, locations, and terminology from the Harry Potter universe.
  verdict : is_member=True  confidence=1.0  -> SELECTED
  tokens  : 'ffindor'[C] 0.36, 'therin'[C] 0.30, 'umbledore'[C] 0.24, '▁Dumbledore'[C] 0.21, '▁Weasley'[C] 0.20, '▁Malfoy'[C] 0.19, '▁Hogwarts'[C] 0.19, '▁Rowling'[C] 0.19, '▁Potter'[C] 0.19, 'Hermione'[C] 0.18

=== feature 88  width=55  rho=53.97 ===
  Gemini  : This vector activates on words related to magic, sorcery, and fantasy folklore.
  verdict : is_member=False  confidence=0.9  -> rejected
  tokens  : '▁magic'[C] 0.36, 'magic'[C] 0.35, 'Magic'[C] 0.34, '▁Magic'[B] 0.33, '▁magical'[C] 0.28, 'magical'[C] 0.26, '▁Magical'[C] 0.23, '▁wizard'[C] 0.18, '▁wizards'[C] 0.17, '▁witchcraft'[C] 0.15

=== feature 68  width=48  rho=16.56 ===
  Gemini  : This feature vector represents the concept of books, reading, and literature.
  verdict : is_member=False  confidence=0.95  -> rejected
  

In [25]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  width={int(widths[feat])}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

1 / 63 candidates have is_member=True

=== feature 46  width=48  rho=129.35 ===
  Gemini  : This vector is related to characters, locations, and terminology from the Harry Potter universe.
  verdict : confidence=1.00  -> SELECTED
  tokens  : 'ffindor'[C] 0.36, 'therin'[C] 0.30, 'umbledore'[C] 0.24, '▁Dumbledore'[C] 0.21, '▁Weasley'[C] 0.20, '▁Malfoy'[C] 0.19, '▁Hogwarts'[C] 0.19, '▁Rowling'[C] 0.19, '▁Potter'[C] 0.19, 'Hermione'[C] 0.18



### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

With variable widths a selected feature can be wider or narrower than the paper's $\lceil 0.01|V^*|\rceil$; its width is printed
with it, and 3.2.f-w compares the $F_C$ widths with the distribution over all $k$ features.

In [26]:
if not F_C:
    print("No feature passed both filters (F_C is empty).")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  width={int(widths[feat])}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

=== feature 46  width=48  rho=129.35  conf=1.00 ===
  This vector is related to characters, locations, and terminology from the Harry Potter universe.
  tokens: 'ffindor'[H], 'therin'[H], 'umbledore'[H], ' Dumbledore'[H], ' Weasley'[H], ' Malfoy'[H], ' Hogwarts'[H], ' Rowling'[H], ' Potter'[H], 'Hermione'[H], ' Hermione'[H], 'demort'[H], ' Voldemort'[H], 'Potter'[H], ' Harry'[H], ' Snape'[H], 'Harry'[H], 'magic'[H], 'Magic'[H], ' magic'[H], ' Draco'[H], 'Hog'[H], ' Severus'[H], ' Magic'[b], ' Gry'[H], ' Hog'[H], ' Ginny'[H], 'otts'[H], 'warts'[H], ' Granger'[H]



In [27]:
# 3.2.f-w Widths of the selected concept features vs. all k features of this fit
w_rank = torch.argsort(torch.argsort(-widths.double(), stable=True), stable=True)   # 0 = widest
print(f"[{VARIANT}] all features: widths min={int(widths.min())}  median={int(widths.median())}  max={int(widths.max())}")
if F_C:
    print(f"F_C widths: " + ", ".join(f"{f}: {int(widths[f])} (rank {int(w_rank[f]) + 1}/{RANK} by width, "
                                      f"{(widths < widths[f]).double().mean():.0%} of features narrower)" for f in F_C))
    print(f"F_C total width = {int(widths[F_C].sum())}   distinct tokens = "
          f"{len({j for f in F_C for j in torch.nonzero(Y[f]).flatten().tolist()})}")
    if len(cand):
        print(f"all rho >= tau candidates: median width = {int(widths[cand['feature'].astype(int).tolist()].median())}")
else:
    print("F_C is empty.")

[m1] all features: widths min=10  median=29  max=239
F_C widths: 46: 48 (rank 24/100 by width, 74% of features narrower)
F_C total width = 48   distinct tokens = 48
all rho >= tau candidates: median width = 28


In [28]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

1 concept features, 48 distinct tokens (C=44, N=1, B=3)


,feature 46
1,ffindor [C] 0.36
2,therin [C] 0.30
3,umbledore [C] 0.24
4,▁Dumbledore [C] 0.21
5,▁Weasley [C] 0.20
6,▁Malfoy [C] 0.19
7,▁Hogwarts [C] 0.19
8,▁Rowling [C] 0.19
9,▁Potter [C] 0.19
10,Hermione [C] 0.18


In [29]:
TOP_N = 30
E_ref = E_vstar_X.float().cpu()  # X-space rows = transform(E_vstar): the space Z lives in
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

=== feature 46   ||z|| = 195.946   sign flipped: True ===

-- C: 1870 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,11879.779297,0.529105,0.361261,True
2,therin,C,10332.246094,0.528831,0.297708,True
3,umbledore,C,9059.843750,0.547460,0.238472,True
4,▁Hogwarts,C,8154.627930,0.678740,0.192877,True
5,▁Dumbledore,C,8027.695312,0.660264,0.208103,True
6,▁Weasley,C,7336.247559,0.567936,0.203925,True
7,▁Malfoy,C,7215.889648,0.595290,0.194578,True
8,▁Hermione,C,7123.187500,0.641894,0.175686,True
9,▁Voldemort,C,7105.919922,0.610808,0.168675,True
10,▁Rowling,C,7003.209961,0.525450,0.188504,True



-- C / B: 2551 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,11879.779297,0.529105,0.361261,True
2,therin,C,10332.246094,0.528831,0.297708,True
3,umbledore,C,9059.843750,0.547460,0.238472,True
4,▁Hogwarts,C,8154.627930,0.678740,0.192877,True
5,▁Dumbledore,C,8027.695312,0.660264,0.208103,True
6,▁Weasley,C,7336.247559,0.567936,0.203925,True
7,▁Malfoy,C,7215.889648,0.595290,0.194578,True
8,▁Hermione,C,7123.187500,0.641894,0.175686,True
9,▁Voldemort,C,7105.919922,0.610808,0.168675,True
10,▁Rowling,C,7003.209961,0.525450,0.188504,True



-- C / B / N: 4780 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,ffindor,C,11879.779297,0.529105,0.361261,True
2,therin,C,10332.246094,0.528831,0.297708,True
3,umbledore,C,9059.843750,0.547460,0.238472,True
4,▁Hogwarts,C,8154.627930,0.678740,0.192877,True
5,▁Dumbledore,C,8027.695312,0.660264,0.208103,True
6,▁Weasley,C,7336.247559,0.567936,0.203925,True
7,▁Malfoy,C,7215.889648,0.595290,0.194578,True
8,▁Hermione,C,7123.187500,0.641894,0.175686,True
9,▁Voldemort,C,7105.919922,0.610808,0.168675,True
10,▁Rowling,C,7003.209961,0.525450,0.188504,True


In [30]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

=== feature 46   ||z|| = 195.946   sign flipped: True ===

-- C: 1870 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,132.996490,61.314606,0.192877,True
2,▁Dumbledore,C,129.376083,62.049290,0.208103,True
3,▁Hermione,C,125.776596,56.633656,0.175686,True
4,▁Voldemort,C,119.685394,59.371643,0.168675,True
5,▁Malfoy,C,116.644760,61.862099,0.194578,True
6,▁Weasley,C,111.284821,65.923157,0.203925,True
7,umbledore,C,107.272659,84.456223,0.238472,True
8,▁Potter,C,106.063316,64.879997,0.188472,True
9,▁Harry,C,105.670853,59.297260,0.162169,True
10,Hermione,C,104.524612,64.814766,0.178623,True



-- C / B: 2551 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,132.996490,61.314606,0.192877,True
2,▁Dumbledore,C,129.376083,62.049290,0.208103,True
3,▁Hermione,C,125.776596,56.633656,0.175686,True
4,▁Voldemort,C,119.685394,59.371643,0.168675,True
5,▁Malfoy,C,116.644760,61.862099,0.194578,True
6,▁Weasley,C,111.284821,65.923157,0.203925,True
7,umbledore,C,107.272659,84.456223,0.238472,True
8,▁Potter,C,106.063316,64.879997,0.188472,True
9,▁Harry,C,105.670853,59.297260,0.162169,True
10,Hermione,C,104.524612,64.814766,0.178623,True



-- C / B / N: 4780 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Hogwarts,C,132.996490,61.314606,0.192877,True
2,▁Dumbledore,C,129.376083,62.049290,0.208103,True
3,▁Hermione,C,125.776596,56.633656,0.175686,True
4,▁Voldemort,C,119.685394,59.371643,0.168675,True
5,▁Malfoy,C,116.644760,61.862099,0.194578,True
6,▁Weasley,C,111.284821,65.923157,0.203925,True
7,umbledore,C,107.272659,84.456223,0.238472,True
8,▁Potter,C,106.063316,64.879997,0.188472,True
9,▁Harry,C,105.670853,59.297260,0.162169,True
10,Hermione,C,104.524612,64.814766,0.178623,True


In [31]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

Label counts among the top 100 of all 4780 tokens in V* (base rate: C=39.1%, B=14.2%, N=46.6%)


,feature,ranking,C,B,N,C in support
0,46,"<e_t, z>",90,9,1,44
1,46,"<e_t / ||e_t||, z>",90,5,5,44


### 3.2.g Extended token set: every sentence of the concept's Wikipedia page, $Z$ fixed
$S_C^{ext} = S_C \cup$ all sentences of the article, $S_N^{ext} = S_N \cup$ as many neutral sentences. The new rows go through the **same** transform $T$ fitted in 3.1.c′ (nothing re-estimated), then only a sparse $Y_{ext}$ is fitted against the frozen X-space $Z$; feature $i$ keeps its 3.1.d width scaled to the larger set, $m_i^{ext} = \mathrm{round}(m_i\,|V^*_{ext}|/|V^*|)$, by $|Y^{dense}_{ext}|$ (refit off, as in 3.1.d). Groups: C / B / N = label in the original $V^*$ (3.2.a), C\* / B\* / N\* = label on the extended sentences; a token gets the first group it matches in the order C, C\*, B, B\*, N, N\*.

In [32]:
# ---- Extended E*: all sentences of the concept's Wikipedia page, Z from 3.1.d fixed, only a sparse Y_ext refit ----
# S_C^ext = S_C + every sentence of the live English Wikipedia article; S_N^ext = S_N + as many neutral sentences
# (one per page of the local wikimedia/wikipedia sample). The new sentences are cached next to RUN_DIR, so re-runs
# use the same page revision and sample.
# Rows go to X-space with the transform T of 3.1.c' exactly as fitted there (same mu_B, W, S_eta: nothing re-estimated),
# then Y_ext = top-m_i per row of (Z^T Z + lambda I)^{-1} Z^T X_ext^T: the Y-step of 3.1.d with Z frozen and the same lambda;
# feature i keeps its 3.1.d width scaled to the larger set, m_i^ext = round(width_i * |V*_ext| / |V*|).
import random
import sys

from experiments.varwidth.common import mask_from_row_widths

sys.path.insert(0, "experiments/resample_sentences")
from fetch_en_sentences import neutral_sentences, page_paragraphs, split_sentences

EXT_PATH = RUN_DIR.parent / "ext_sentences.json"   # shared by the five variable-width variants of this concept
EXT_GROUPS = ["C", "C*", "B", "B*", "N", "N*"]

# 1) sentences
ext = load_json(EXT_PATH, None)
if ext is None:
    canon, revid, paras = page_paragraphs(CONCEPT)
    page_sents = split_sentences(paras)
    ext = {"title": canon, "revid": revid, "concept": page_sents,
           "neutral": [r["sentence"] for r in neutral_sentences(
               Path("experiments/resample_sentences/data/wiki_dump_pages.jsonl"), len(page_sents),
               random.Random(SEED), exclude=set(page_sents))]}
    save_json(EXT_PATH, ext)
ext_concept = list(dict.fromkeys([s for s, lbl in dataset if lbl == CONCEPT] + ext["concept"]))
ext_neutral = list(dict.fromkeys([s for s, lbl in dataset if lbl != CONCEPT] + ext["neutral"]))
print(f"'{ext['title']}' rev {ext['revid']}: {len(ext['concept'])} page sentences, {len(ext['neutral'])} new neutral "
      f"({EXT_PATH})\nS_C^ext = {len(ext_concept)} sentences   S_N^ext = {len(ext_neutral)} sentences")

# 2) extended token set (a superset of V*) and its groups
special_ext = set(tokenizer.all_special_ids)


def token_set(sents):
    return {int(t) for s in sents for t in tokenizer(s, add_special_tokens=False)["input_ids"]} - special_ext


c_ids, n_ids = token_set(ext_concept), token_set(ext_neutral)
vstar_ext = sorted(c_ids | n_ids | set(vstar_ids))
codes_ext = np.array([2 if t in c_ids and t in n_ids else 1 if t in c_ids else 0 for t in vstar_ext])
orig_code = dict(zip(vstar_ids, codes.tolist()))


def ext_group(t: int, c_ext: int) -> str:
    """First match in the order C, C*, B, B*, N, N* (C/B/N: label in V*, starred: label on the extended sentences)."""
    o = orig_code.get(t)
    for g, hit in [("C", o == 1), ("C*", c_ext == 1), ("B", o == 2), ("B*", c_ext == 2), ("N", o == 0), ("N*", c_ext == 0)]:
        if hit:
            return g


groups_ext = np.array([ext_group(t, int(c)) for t, c in zip(vstar_ext, codes_ext)])
print(f"|V*_ext| = {len(vstar_ext)}  (|V*| = {len(vstar_ids)}, {len(vstar_ext) - len(vstar_ids)} new)")
display(pd.Series(groups_ext).value_counts().reindex(EXT_GROUPS).fillna(0).astype(int).rename("tokens").to_frame().T)

# 3) X_ext = T(E_{V*_ext}): the 3.1.c' transform, unchanged
ids_ext = torch.tensor(vstar_ext)
E_ext = E.index_select(0, ids_ext.to(E.device)).detach().float().cpu()
X_ext = transform(E_ext)                                                   # (|V*_ext|, d), X-space
pos_ext = {t: j for j, t in enumerate(vstar_ext)}
in_vstar = [pos_ext[t] for t in vstar_ids]
print(f"check: max |X_ext[V*] - E_vstar_X| = {(X_ext[in_vstar] - E_vstar_X.float().cpu()).abs().max():.2e}")

# 4) Y-step with Z frozen (dense ridge solve as in fit_varwidth, then each row's top m_i^ext; no refit, as CFG.refit=False)
A_ext = X_ext.T.contiguous().to(DEVICE)                                    # (d, |V*_ext|)
Z_dev = Z.to(DEVICE)
with Timer() as t_ext:
    inv_Z = torch.linalg.inv(Z_dev.T @ Z_dev + torch.eye(Z_dev.shape[1], device=DEVICE) * ridge_used)
    G_ext = A_ext.T @ Z_dev @ inv_Z                                        # (|V*_ext|, k), the G_ layout
    m_ext = (widths.double() * len(vstar_ext) / len(vstar_ids)).round().long().clamp(1, len(vstar_ext))
    Yd_ext = G_ext.T.contiguous()                                          # (k, |V*_ext|) dense
    Y_ext = (Yd_ext * mask_from_row_widths(Yd_ext.abs(), m_ext.to(Yd_ext.device))).float().cpu()   # (k, |V*_ext|)
A_ext_cpu = A_ext.cpu()
ext_err = (torch.norm(A_ext_cpu - Z @ Y_ext) / torch.norm(A_ext_cpu)).item()
print(f"fixed-Z fit: {t_ext['elapsed']:.3f}s, each feature keeps its 3.1.d width x |V*_ext|/|V*|: "
      f"{int(m_ext.min())}-{int(m_ext.max())} tokens per feature, {int(m_ext.sum())} in total")
print(f"relative error: on X_ext (fixed Z) = {ext_err:.4f}   3.1.d joint fit on A = {rel_err:.4f}")
del A_ext, Z_dev, G_ext, Yd_ext, A_ext_cpu, E_ext

'Harry Potter' rev 1379437311: 403 page sentences, 403 new neutral (runs/variable_width/Harry_Potter/ext_sentences.json)
S_C^ext = 444 sentences   S_N^ext = 682 sentences
|V*_ext| = 7408  (|V*| = 4780, 2628 new)


,C,C*,B,B*,N,N*
tokens,1870,465,681,158,2138,2096


check: max |X_ext[V*] - E_vstar_X| = 0.00e+00
fixed-Z fit: 0.016s, each feature keeps its 3.1.d width x |V*_ext|/|V*|: 15-370 tokens per feature, 7441 in total
relative error: on X_ext (fixed Z) = 0.8773   3.1.d joint fit on A = 0.8510


In [33]:
# ---- Rate every token of V*_ext on each concept feature, then list the feature's Y_ext support (X-space, where Z lives) ----
#   <x_t, z~>            inner product of the transformed token row with the feature direction
#   <x_t / ||x_t||, z~>  the same with the token row normalized
#   words                whole words of S_C^ext and S_N^ext the token was cut from (most frequent first)
# z~'s sign is arbitrary (Y absorbs it): oriented as in 3.2.f, so its 3.1 support tokens mostly have positive Y.
# Support table: tokens with Y_ext[feat, t] != 0, by decreasing |Y_ext|; group = first match in C, C*, B, B*, N, N* (3.2.g);
# in V*: the token is in the original token set;  in original support: Y[feat, t] != 0 in the 3.1 fit.
import re
from collections import Counter, defaultdict

SHOW_ROWS = 100   # rows shown per ranking; None shows all |V*_ext| tokens (the full tables are in ext_rating)

WORD_RE = re.compile(r"\w+(?:['’-]\w+)*")
token_words_ext = defaultdict(Counter)  # token id -> Counter of source words
for sent in ext_concept + ext_neutral:
    enc = tokenizer(sent, add_special_tokens=False, return_offsets_mapping=True)
    words = [(m.start(), m.end(), m.group()) for m in WORD_RE.finditer(sent)]
    for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
        for ws, we, w in words:
            if ws < e and s < we:  # token's character span overlaps this word
                token_words_ext[tid][w] += 1
words_ext = [[w for w, _ in token_words_ext[t].most_common()] for t in vstar_ext]

ext_strs = tokenizer.convert_ids_to_tokens(vstar_ext)
x_norms = X_ext.norm(dim=1)
vstar_set = set(vstar_ids)
ext_rating, ext_support = {}, {}
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = X_ext @ z
    orig_sup = {vstar_ids[j] for j in torch.nonzero(y).flatten().tolist()}
    ext_sup = (Y_ext[feat] != 0).numpy()

    # 1) every token of V*_ext, ranked by the two inner products
    # support: first match of (original: in this direction's 3.1 support, extended: in its Y_ext support, none)
    df = pd.DataFrame({
        "token": ext_strs,
        "group": groups_ext,
        "words": words_ext,
        "support": ["original" if t in orig_sup else "extended" if e else "none" for t, e in zip(vstar_ext, ext_sup)],
        "<x_t, z>": dots.numpy(),
        "<x_t / ||x_t||, z>": (dots / x_norms).numpy(),
        "||x_t||": x_norms.numpy(),
        "Y_ext": (sign * Y_ext[feat]).numpy(),
        "in Y_ext support": (Y_ext[feat] != 0).numpy(),
    })
    ext_rating[feat] = df

    print(f"{'=' * 100}\n=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0}   "
          f"|V*_ext| = {len(df)}   Y_ext support = {int(df['in Y_ext support'].sum())} ===\n{'=' * 100}")
    by_group = df.groupby("group")[["<x_t, z>", "<x_t / ||x_t||, z>"]].agg(["mean", "median", "max"])
    by_group.insert(0, "tokens", df["group"].value_counts())
    display(by_group.reindex(EXT_GROUPS).style.format("{:.3f}", subset=by_group.columns[1:]))
    for col in ["<x_t, z>", "<x_t / ||x_t||, z>"]:
        ranked = df.sort_values(col, ascending=False, ignore_index=True)
        ranked.index += 1
        print(f"\n-- all {len(df)} tokens of V*_ext by {col} (top {SHOW_ROWS or len(df)} shown) --")
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(ranked if SHOW_ROWS is None else ranked.head(SHOW_ROWS))

    # 2) the feature's Y_ext support, by decreasing |Y_ext|
    sup = torch.nonzero(Y_ext[feat]).flatten()
    sup = sup[torch.argsort(Y_ext[feat, sup].abs(), descending=True)].tolist()
    sup_df = pd.DataFrame({
        "token": [ext_strs[j] for j in sup],
        "group": groups_ext[sup],
        "words": [words_ext[j] for j in sup],
        "Y_ext": (sign * Y_ext[feat, sup]).numpy(),
        "in V*": [vstar_ext[j] in vstar_set for j in sup],
        "in original support": [vstar_ext[j] in orig_sup for j in sup],
        "<x_t, z>": dots[sup].numpy(),
        "<x_t / ||x_t||, z>": (dots[sup] / x_norms[sup]).numpy(),
    }, index=pd.RangeIndex(1, len(sup) + 1, name="rank"))
    ext_support[feat] = sup_df

    lost = sorted(orig_sup - {vstar_ext[j] for j in sup})
    counts = "  ".join(f"{g}: {int((sup_df['group'] == g).sum())}" for g in EXT_GROUPS)
    print(f"\n-- Y_ext support: {len(sup_df)} tokens [{counts}] | in V*: {int(sup_df['in V*'].sum())} | "
          f"in original support: {int(sup_df['in original support'].sum())}/{len(orig_sup)} --")
    if lost:
        print(f"original support tokens no longer in the support ({len(lost)}): {tokenizer.convert_ids_to_tokens(lost)}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(sup_df.style.format({"Y_ext": "{:+.4f}", "<x_t, z>": "{:.2f}", "<x_t / ||x_t||, z>": "{:.3f}"}))

=== feature 46   ||z|| = 195.946   sign flipped: True   |V*_ext| = 7408   Y_ext support = 74 ===



-- all 7408 tokens of V*_ext by <x_t, z> (top 100 shown) --


,token,group,words,support,"<x_t, z>","<x_t / ||x_t||, z>",||x_t||,Y_ext,in Y_ext support
1,ffindor,C,[Gryffindor],original,11879.779297,103.676102,114.585510,0.361261,True
2,therin,C,[Slytherin's],original,10332.246094,103.622375,99.710571,0.297708,True
3,umbledore,C,"[Dumbledore's, Dumbledore]",original,9059.843750,107.272659,84.456223,0.238472,True
4,▁Hogwarts,C,[Hogwarts],original,8154.627930,132.996506,61.314606,0.192877,True
5,▁Dumbledore,C,"[Dumbledore, Dumbledore's]",original,8027.695312,129.376099,62.049290,0.208103,True
6,▁Weasley,C,[Weasley],original,7336.247559,111.284836,65.923157,0.203925,True
7,▁Malfoy,C,[Malfoy],original,7215.889648,116.644760,61.862099,0.194578,True
8,▁Hermione,C,"[Hermione, Hermione's]",original,7123.187500,125.776581,56.633656,0.175686,True
9,▁Voldemort,C,"[Voldemort, Voldemort's]",original,7105.919922,119.685417,59.371643,0.168675,True
10,▁Rowling,C,"[Rowling, Rowling's]",original,7003.209961,102.959953,68.018776,0.188504,True



-- all 7408 tokens of V*_ext by <x_t / ||x_t||, z> (top 100 shown) --


,token,group,words,support,"<x_t, z>","<x_t / ||x_t||, z>",||x_t||,Y_ext,in Y_ext support
1,▁Hogwarts,C,[Hogwarts],original,8154.627930,132.996506,61.314606,0.192877,True
2,▁Dumbledore,C,"[Dumbledore, Dumbledore's]",original,8027.695312,129.376099,62.049290,0.208103,True
3,▁Hermione,C,"[Hermione, Hermione's]",original,7123.187500,125.776581,56.633656,0.175686,True
4,▁Voldemort,C,"[Voldemort, Voldemort's]",original,7105.919922,119.685417,59.371643,0.168675,True
5,▁Malfoy,C,[Malfoy],original,7215.889648,116.644760,61.862099,0.194578,True
6,▁Weasley,C,[Weasley],original,7336.247559,111.284836,65.923157,0.203925,True
7,umbledore,C,"[Dumbledore's, Dumbledore]",original,9059.843750,107.272659,84.456223,0.238472,True
8,▁Potter,C,"[Potter, Potter's, Potter-themed, Pottermore, Potter's's]",original,6881.388184,106.063324,64.879997,0.188472,True
9,▁Harry,C,"[Harry, Harry's, Harryplax]",original,6265.991211,105.670837,59.297260,0.162169,True
10,Hermione,C,[Hermione],original,6774.737305,104.524597,64.814766,0.178623,True



-- Y_ext support: 74 tokens [C: 59  C*: 5  B: 5  B*: 1  N: 2  N*: 2] | in V*: 66 | in original support: 48/48 --


,token,group,words,Y_ext,in V*,in original support,,
rank,,,,,,,,
1,ffindor,C,['Gryffindor'],+0.3613,True,True,11879.78,103.676
2,therin,C,"[""Slytherin's""]",+0.2977,True,True,10332.25,103.622
3,umbledore,C,"[""Dumbledore's"", 'Dumbledore']",+0.2385,True,True,9059.84,107.273
4,▁Dumbledore,C,"['Dumbledore', ""Dumbledore's""]",+0.2081,True,True,8027.70,129.376
5,▁Weasley,C,['Weasley'],+0.2039,True,True,7336.25,111.285
6,▁Malfoy,C,['Malfoy'],+0.1946,True,True,7215.89,116.645
7,▁Hogwarts,C,['Hogwarts'],+0.1929,True,True,8154.63,132.997
8,▁Rowling,C,"['Rowling', ""Rowling's""]",+0.1885,True,True,7003.21,102.960
9,▁Potter,C,"['Potter', ""Potter's"", 'Potter-themed', 'Pottermore', ""Potter's's""]",+0.1885,True,True,6881.39,106.063


In [34]:
# ---- Dense refit: fix Z, solve Y' = argmin ||E*^T - Z Y'||_F (no sparsity) ----
TOP_DIRS = 5
HP_FEATS = list(F_C)          # the Harry Potter concept column(s) of Z; override e.g. [37] if you want one
HP_MARK = "★★★ HP ★★★"

A_ref = E_vstar_X.double().cpu().T                                                # (d, |V*|), X-space
Z64 = Z.double()                                                                    # (d, k)
Y_dense = torch.linalg.lstsq(Z64, A_ref).solution.float()                           # (k, |V*|)

dense_err = (torch.norm(A_ref - Z64 @ Y_dense.double()) / torch.norm(A_ref)).item()
sparse_err = (torch.norm(A_ref - Z64 @ Y.double()) / torch.norm(A_ref)).item()
print(f"relative error: sparse Y = {sparse_err:.4f}   dense Y' = {dense_err:.4f}")
print(f"HP feature(s): {HP_FEATS}   (||z|| = {[round(Z[:, f].norm().item(), 3) for f in HP_FEATS]})\n")

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    print(f"{'=' * 100}\n=== HP feature {hp}: {len(support)} tokens with nonzero Y, by decreasing |Y| ===\n{'=' * 100}")
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        top = order[:TOP_DIRS]

        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}")
        for r, f in enumerate(top, 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        for f in HP_FEATS:
            if f not in top:
                print(f"       ..  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}   <-- {HP_MARK} (rank {order.index(f) + 1}/{len(order)})")

relative error: sparse Y = 0.8510   dense Y' = 0.8107
HP feature(s): [46]   (||z|| = [195.946])

=== HP feature 46: 48 tokens with nonzero Y, by decreasing |Y| ===

  1. 'ffindor' [C]   sparse Y[46] = -0.361
       #1  z_46   Y' = -0.361   <-- ★★★ HP ★★★
       #2  z_94   Y' = +0.119
       #3  z_51   Y' = -0.116
       #4  z_88   Y' = +0.080
       #5  z_85   Y' = +0.039

  2. 'therin' [C]   sparse Y[46] = -0.298
       #1  z_46   Y' = -0.298   <-- ★★★ HP ★★★
       #2  z_51   Y' = -0.071
       #3  z_88   Y' = +0.053
       #4  z_91   Y' = +0.046
       #5  z_80   Y' = +0.043

  3. 'umbledore' [C]   sparse Y[46] = -0.238
       #1  z_46   Y' = -0.238   <-- ★★★ HP ★★★
       #2  z_83   Y' = -0.040
       #3  z_8    Y' = -0.035
       #4  z_94   Y' = -0.033
       #5  z_44   Y' = +0.032

  4. '▁Dumbledore' [C]   sparse Y[46] = -0.208
       #1  z_46   Y' = -0.208   <-- ★★★ HP ★★★
       #2  z_71   Y' = +0.021
       #3  z_60   Y' = +0.021
       #4  z_37   Y' = -0.019
       #5  z_20  

In [35]:
# ---- Tokens where the HP direction dominates Y': top-1 is HP and |Y'_1| >= RATIO * |Y'_2| ----
RATIO = 2.0

for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    kept = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] == hp and first >= RATIO * second:
            kept.append((rank_t, j, tok, order, first / second))

    print(f"{'=' * 100}\n=== HP feature {hp}: {len(kept)} / {len(support)} tokens where z_{hp} is top-1 "
          f"and |Y'| >= {RATIO:g}x the runner-up ===\n{'=' * 100}")
    for rank_t, j, tok, order, ratio in kept:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   "
              f"top-1/top-2 = {ratio:.2f}x")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")

=== HP feature 46: 40 / 48 tokens where z_46 is top-1 and |Y'| >= 2x the runner-up ===

  1. 'ffindor' [C]   sparse Y[46] = -0.361   top-1/top-2 = 3.05x
       #1  z_46   Y' = -0.361   <-- ★★★ HP ★★★
       #2  z_94   Y' = +0.119
       #3  z_51   Y' = -0.116
       #4  z_88   Y' = +0.080
       #5  z_85   Y' = +0.039

  2. 'therin' [C]   sparse Y[46] = -0.298   top-1/top-2 = 4.19x
       #1  z_46   Y' = -0.298   <-- ★★★ HP ★★★
       #2  z_51   Y' = -0.071
       #3  z_88   Y' = +0.053
       #4  z_91   Y' = +0.046
       #5  z_80   Y' = +0.043

  3. 'umbledore' [C]   sparse Y[46] = -0.238   top-1/top-2 = 5.93x
       #1  z_46   Y' = -0.238   <-- ★★★ HP ★★★
       #2  z_83   Y' = -0.040
       #3  z_8    Y' = -0.035
       #4  z_94   Y' = -0.033
       #5  z_44   Y' = +0.032

  4. '▁Dumbledore' [C]   sparse Y[46] = -0.208   top-1/top-2 = 9.72x
       #1  z_46   Y' = -0.208   <-- ★★★ HP ★★★
       #2  z_71   Y' = +0.021
       #3  z_60   Y' = +0.021
       #4  z_37   Y' = -0.019
      

In [36]:
# ---- Tokens skipped by the previous cell: HP is not top-1, or top-1 but < RATIO x the runner-up ----
for hp in HP_FEATS:
    mags = Y[hp].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    skipped = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] != hp:
            skipped.append((rank_t, j, tok, order, f"HP is not top-1 (rank {order.index(hp) + 1}/{len(order)})"))
        elif first < RATIO * second:
            skipped.append((rank_t, j, tok, order, f"HP is top-1 but only {first / second:.2f}x the runner-up"))

    n_not_top = sum(r.startswith("HP is not") for *_, r in skipped)
    print(f"{'=' * 100}\n=== HP feature {hp}: {len(skipped)} / {len(support)} tokens skipped "
          f"({n_not_top} HP not top-1, {len(skipped) - n_not_top} top-1 but < {RATIO:g}x) ===\n{'=' * 100}")
    for rank_t, j, tok, order, reason in skipped:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{hp}] = {Y[hp, j]:+.3f}   -> {reason}")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

=== HP feature 46: 8 / 48 tokens skipped (8 HP not top-1, 0 top-1 but < 2x) ===

 18. 'magic' [C]   sparse Y[46] = +0.143   -> HP is not top-1 (rank 2/100)
       #1  z_88   Y' = -0.346
       #2  z_46   Y' = +0.143   <-- ★★★ HP ★★★
       #3  z_66   Y' = +0.016
       #4  z_39   Y' = +0.015
       #5  z_15   Y' = +0.013

 19. 'Magic' [C]   sparse Y[46] = +0.138   -> HP is not top-1 (rank 2/100)
       #1  z_88   Y' = -0.340
       #2  z_46   Y' = +0.138   <-- ★★★ HP ★★★
       #3  z_65   Y' = -0.015
       #4  z_36   Y' = +0.014
       #5  z_32   Y' = +0.012

 20. '▁magic' [C]   sparse Y[46] = +0.133   -> HP is not top-1 (rank 2/100)
       #1  z_88   Y' = -0.362
       #2  z_46   Y' = +0.133   <-- ★★★ HP ★★★
       #3  z_19   Y' = -0.013
       #4  z_86   Y' = +0.013
       #5  z_66   Y' = -0.010

 24. '▁Magic' [B]   sparse Y[46] = +0.121   -> HP is not top-1 (rank 2/100)
       #1  z_88   Y' = -0.333
       #2  z_46   Y' = +0.121   <-- ★★★ HP ★★★
       #3  z_32   Y' = +0.014
      

In [37]:
# ---- All C-labelled tokens, sorted by |Y'_HP| / max_{i != HP} |Y'_i| (= top-1/top-2 when HP is top-1) ----
for hp in HP_FEATS:
    c_cols = np.where(codes == 1)[0].tolist()
    rows = []
    for j in c_cols:
        y_dense_t = Y_dense[:, j]
        others_max = torch.cat([y_dense_t[:hp], y_dense_t[hp + 1:]]).abs().max().item()
        rows.append((y_dense_t[hp].abs().item() / others_max, j))
    rows.sort(reverse=True)

    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for _, j in rows])
    n_pass = sum(r >= RATIO for r, _ in rows)
    n_top1 = sum(r >= 1.0 for r, _ in rows)
    print(f"{'=' * 100}\n=== HP feature {hp}: {len(rows)} C tokens | HP top-1: {n_top1} | "
          f"HP top-1 and >= {RATIO:g}x: {n_pass} ===\n{'=' * 100}")

    shown_ratio_line = shown_top1_line = False
    for rank_t, ((ratio, j), tok) in enumerate(zip(rows, toks), 1):
        if not shown_ratio_line and ratio < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and ratio < 1.0:
            print(f"\n{'-' * 36} HP no longer top-1 {'-' * 36}")
            shown_top1_line = True

        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        in_support = "in HP support" if Y[hp, j] != 0 else "not in HP support"
        print(f"\n{rank_t:3d}. {tok!r} [C]   ratio = {ratio:.2f}x   sparse Y[{hp}] = {Y[hp, j]:+.3f}  ({in_support})")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {HP_MARK}" if f in HP_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if hp not in order[:TOP_DIRS]:
            print(f"       ..  z_{hp:<4d} Y' = {y_dense_t[hp]:+.3f}   <-- {HP_MARK} (rank {order.index(hp) + 1}/{len(order)})")

=== HP feature 46: 1870 C tokens | HP top-1: 95 | HP top-1 and >= 2x: 41 ===

  1. '▁Dumbledore' [C]   ratio = 9.72x   sparse Y[46] = -0.208  (in HP support)
       #1  z_46   Y' = -0.208   <-- ★★★ HP ★★★
       #2  z_71   Y' = +0.021
       #3  z_60   Y' = +0.021
       #4  z_37   Y' = -0.019
       #5  z_20   Y' = +0.019

  2. '▁Malfoy' [C]   ratio = 9.52x   sparse Y[46] = -0.195  (in HP support)
       #1  z_46   Y' = -0.195   <-- ★★★ HP ★★★
       #2  z_96   Y' = +0.020
       #3  z_89   Y' = +0.020
       #4  z_52   Y' = +0.018
       #5  z_44   Y' = -0.018

  3. '▁Potter' [C]   ratio = 9.39x   sparse Y[46] = -0.188  (in HP support)
       #1  z_46   Y' = -0.188   <-- ★★★ HP ★★★
       #2  z_71   Y' = -0.020
       #3  z_44   Y' = -0.019
       #4  z_60   Y' = +0.017
       #5  z_47   Y' = +0.017

  4. '▁Hermione' [C]   ratio = 8.85x   sparse Y[46] = -0.176  (in HP support)
       #1  z_46   Y' = -0.176   <-- ★★★ HP ★★★
       #2  z_27   Y' = -0.020
       #3  z_88   Y' = -0.016
 

In [38]:
# ---- How orthogonal is each HP direction to the span of the other Z columns? ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    Z64 = Z.double()
    k = Z64.shape[1]


    def angle_to_span(f: int, exclude=()) -> dict:
        """Angle between z_f and span{z_i : i != f, i not in exclude}."""
        others = [i for i in range(k) if i != f and i not in exclude]
        z = Z64[:, f]
        coef = torch.linalg.lstsq(Z64[:, others], z.unsqueeze(1)).solution.squeeze(1)
        proj = Z64[:, others] @ coef
        resid = z - proj
        sin = (resid.norm() / z.norm()).item()                  # 1 = orthogonal, 0 = inside the span
        cos_single = F.cosine_similarity(Z64[:, others], z.unsqueeze(1), dim=0)
        best = int(cos_single.abs().argmax())
        return {
            "feature": f,
            "angle_deg": math.degrees(math.asin(min(sin, 1.0))),
            "resid/||z||": sin,
            "proj/||z||": (proj.norm() / z.norm()).item(),
            "max |cos| to one z_i": cos_single.abs().max().item(),
            "closest z_i": others[best],
            "top-3 lstsq coef (i: c)": ", ".join(
                f"{others[i]}: {coef[i]:+.2f}" for i in torch.argsort(coef.abs(), descending=True)[:3].tolist()
            ),
        }


    print(f"d = {Z64.shape[0]}, k = {k}  ->  span of the other {k - 1} columns is a {k - 1}-dim subspace of R^{Z64.shape[0]}")
    print("For reference, a random direction would sit at about "
          f"{math.degrees(math.acos(math.sqrt((k - 1) / Z64.shape[0]))):.1f}° from a random {k - 1}-dim subspace\n")

    hp_rows = [angle_to_span(f) for f in HP_FEATS]
    if len(HP_FEATS) > 1:  # also measure against non-HP directions only
        hp_rows += [{**angle_to_span(f, exclude=HP_FEATS), "feature": f"{f} (vs non-HP only)"} for f in HP_FEATS]
    display(pd.DataFrame(hp_rows).set_index("feature"))

    # Baseline: same measurement for every feature, to see whether HP is unusual
    all_angles = pd.Series({f: angle_to_span(f)["angle_deg"] for f in range(k)}).sort_values()
    print("\nangle to span of the others, all features:")
    print(all_angles.describe().round(2).to_string())
    for f in HP_FEATS:
        pct = (all_angles < all_angles[f]).mean()
        print(f"HP feature {f}: {all_angles[f]:.2f}°  ->  more orthogonal than {pct:.0%} of features "
              f"(rank {all_angles.index.get_loc(f) + 1}/{k}, ascending)")


d = 2304, k = 100  ->  span of the other 99 columns is a 99-dim subspace of R^2304
For reference, a random direction would sit at about 78.0° from a random 99-dim subspace



,angle_deg,resid/||z||,proj/||z||,max |cos| to one z_i,closest z_i,top-3 lstsq coef (i: c)
feature,,,,,,
46,54.056665,0.809598,0.586985,0.525653,88,"88: +0.39, 30: -0.12, 68: -0.07"



angle to span of the others, all features:
count    100.00
mean      64.53
std        9.29
min       36.80
25%       57.67
50%       66.83
75%       71.85
max       78.98
HP feature 46: 54.06°  ->  more orthogonal than 12% of features (rank 13/100, ascending)


In [39]:
# ---- The 5 directions closest to the HP direction: angle + Gemini description ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    HP = HP_FEATS[0]
    N_NEIGHBORS = 5
    N_TOKENS_SHOW = 15

    z_hp = Z[:, HP]
    cos_all = F.cosine_similarity(Z, z_hp.unsqueeze(1), dim=0)        # (k,)
    cos_all[HP] = 0.0
    neighbors = torch.argsort(cos_all.abs(), descending=True)[:N_NEIGHBORS].tolist()


    def describe(f: int) -> str:
        if f not in descriptions:  # only tau-passing features were described in Step 3.2
            descriptions[f] = get_description(client, judge_input(Y, f, feat_tokens.loc[f, "activating_tokens"]), sys_prompt=DESC_SYS)
            save_judge()
        return descriptions[f]


    def top_tokens(f: int) -> str:
        mags = Y[f].abs()
        top = torch.argsort(mags, descending=True)[:N_TOKENS_SHOW]
        top = top[mags[top] > 0].tolist()
        toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
        return ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}]" for t, j in zip(toks, top))


    print(f"=== HP feature {HP} ===\n  Gemini : {describe(HP)}\n  tokens : {top_tokens(HP)}\n")

    rows = []
    for r, f in enumerate(neighbors, 1):
        c = cos_all[f].item()
        angle = math.degrees(math.acos(min(abs(c), 1.0)))   # angle between the lines, 0°..90°
        shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
        rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                     "overlap cos²": c ** 2, "shared tokens": shared, "description": describe(f)})
        print(f"#{r}  feature {f}   angle = {angle:.1f}°   cos = {c:+.3f}   (cos² = {c ** 2:.0%} of z_{HP} along it)   "
              f"shared support tokens = {shared}")
        print(f"  Gemini : {describe(f)}")
        print(f"  tokens : {top_tokens(f)}\n")


=== HP feature 46 ===
  Gemini : This vector is related to characters, locations, and terminology from the Harry Potter universe.
  tokens : 'ffindor'[C], 'therin'[C], 'umbledore'[C], '▁Dumbledore'[C], '▁Weasley'[C], '▁Malfoy'[C], '▁Hogwarts'[C], '▁Rowling'[C], '▁Potter'[C], 'Hermione'[C], '▁Hermione'[C], 'demort'[C], '▁Voldemort'[C], 'Potter'[C], '▁Harry'[C]

#1  feature 88   angle = 58.3°   cos = +0.526   (cos² = 28% of z_46 along it)   shared support tokens = 10
  Gemini : This vector activates on words related to magic, sorcery, and fantasy folklore.
  tokens : '▁magic'[C], 'magic'[C], 'Magic'[C], '▁Magic'[B], '▁magical'[C], 'magical'[C], '▁Magical'[C], '▁wizard'[C], '▁wizards'[C], '▁witchcraft'[C], '▁Wizard'[C], 'wizard'[C], '▁Wizards'[C], '▁witch'[C], '▁witches'[C]

#2  feature 30   angle = 82.1°   cos = -0.138   (cos² = 2% of z_46 along it)   shared support tokens = 0
  Gemini : This feature vector represents concepts related to formal education, schools, and academic instructio

In [40]:
# ---- The 5 directions most orthogonal to the HP direction ----
if not F_C:
    print("F_C is empty: nothing to analyse in this cell.")
else:
    N_ORTHO = 5

    others = torch.tensor([f for f in range(Z.shape[1]) if f != HP])
    most_ortho = others[torch.argsort(cos_all[others].abs())[:N_ORTHO]].tolist()

    rows = []
    for r, f in enumerate(most_ortho, 1):
        c = cos_all[f].item()
        angle = math.degrees(math.acos(min(abs(c), 1.0)))
        shared = int(((Y[HP] != 0) & (Y[f] != 0)).sum())
        rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                     "shared tokens": shared, "description": describe(f)})
        print(f"#{r}  feature {f}   angle = {angle:.2f}°   cos = {c:+.4f}   shared support tokens = {shared}")
        print(f"  Gemini : {describe(f)}")
        print(f"  tokens : {top_tokens(f)}\n")

    display(pd.DataFrame(rows).set_index("rank"))


#1  feature 35   angle = 89.98°   cos = -0.0004   shared support tokens = 0
  Gemini : This vector is related to exemplification, demonstration, and pointing out similarities or specific instances.
  tokens : '▁such'[B], 'such'[N], '▁these'[B], '▁like'[B], '▁this'[B], 'These'[B], '▁tales'[C], '▁including'[B], '▁example'[C], 'Like'[B], '▁examples'[C], '▁instance'[C], '▁likes'[C], 'including'[C], '▁those'[C]

#2  feature 69   angle = 89.96°   cos = +0.0007   shared support tokens = 0
  Gemini : This vector represents temporal sequencing and the relative timing of events or actions.
  tokens : '▁late'[N], '▁early'[B], '▁later'[B], '▁Later'[C], 'late'[N], '▁follows'[C], '▁follow'[C], '▁earlier'[C], '▁following'[B], '▁followed'[C], '▁followers'[C], 'Soon'[C], '▁eventually'[C], '▁shortly'[C], 'Shortly'[N]

#3  feature 59   angle = 89.95°   cos = -0.0009   shared support tokens = 0
  Gemini : This vector represents concepts related to methods, means, and ways of accomplishing something.
  tok

,feature,angle_deg,cos (signed),shared tokens,description
rank,,,,,
1,35,89.975968,-0.000419,0,"This vector is related to exemplification, demonstration, and pointing out similarities or specific instances."
2,69,89.959727,0.000703,0,This vector represents temporal sequencing and the relative timing of events or actions.
3,59,89.946813,-0.000928,0,"This vector represents concepts related to methods, means, and ways of accomplishing something."
4,41,89.939170,-0.001062,0,This vector represents personal and expletive subject pronouns in various grammatical cases and capitalizations.
5,1,89.902444,0.001703,0,"This vector represents concepts of prior occurrence, sequence, or historical precedence."


---
### 3.2.w Original vs transformed: matched-feature overlap

Do the two fits find the same token groups? Each feature is compared through its **support**, the tokens it uses:
$S^0_i = \{t : Y^0_{i,t}\neq 0\}$ (original fit) and $S_j = \{t : Y_{j,t}\neq 0\}$ (transformed fit).

- Both fits use the same method (`CFG`), so both have $K$ nonzeros in total; with `fixed` every row has exactly
  $\lceil s|V^*|\rceil$, otherwise the widths differ per feature and between the fits. Jaccard handles unequal sizes.
- $J_{ij} = |S^0_i\cap S_j| \,/\, |S^0_i\cup S_j|$ for all $k\times k$ pairs. Feature indices are arbitrary in each fit, so
  features are paired by the one-to-one matching with the largest total Jaccard (Hungarian algorithm,
  `scipy.optimize.linear_sum_assignment`). `best Jaccard` is the best partner without the one-to-one constraint.
- Directions: a transformed feature lives in X-space. `T.unmap` ($\tilde{\mathbf z}\,S_\eta^{-\gamma/2}W^{-1}$) maps it to the raw space,
  where it is compared with $\mathbf z^0_i$ by $|\cos|$ (the sign of a feature is arbitrary).
- **`F_C0` here is the list of $\rho\ge\tau$ candidates of the original fit (3.2.e′), not a Gemini-judged selection** (no Gemini
  calls are made for the original fit). The column `orig in F_C0` therefore means "original feature passes $\rho\ge\tau$".

In [41]:
from scipy.optimize import linear_sum_assignment

S0 = (Y0 != 0).numpy()                    # (k, |V*|) supports, original fit
S1 = (Y != 0).numpy()                     # (k, |V*|) supports, transformed fit
print(f"|S0_i| (original widths): min={S0.sum(1).min()} median={int(np.median(S0.sum(1)))} max={S0.sum(1).max()}   "
      f"|S_j| (transformed widths): min={S1.sum(1).min()} median={int(np.median(S1.sum(1)))} max={S1.sum(1).max()}")

inter = S0.astype(np.int64) @ S1.T.astype(np.int64)            # (k, k): |S0_i ∩ S_j|
union = S0.sum(1)[:, None] + S1.sum(1)[None, :] - inter
J = inter / np.maximum(union, 1)

rows0, cols1 = linear_sum_assignment(-J)                       # maximize total Jaccard
match0 = dict(zip(rows0.tolist(), cols1.tolist()))             # original i -> transformed j
match1 = {j: i for i, j in match0.items()}                     # transformed j -> original i

Z_back = T.unmap(Z.T).T.float()                                # (d, k): transformed features in the raw space
cos_map = F.cosine_similarity(Z0[:, rows0], Z_back[:, cols1], dim=0).abs().numpy()

match_df = pd.DataFrame({
    "orig feature": rows0,
    "transformed feature": cols1,
    "|S0|": S0.sum(1)[rows0],
    "|S|": S1.sum(1)[cols1],
    "|S0∩S|": inter[rows0, cols1],
    "Jaccard": J[rows0, cols1],
    "best Jaccard": J.max(1)[rows0],
    "best partner": J.argmax(1)[rows0],
    "|cos(z0, unmap(z~))|": cos_map,
    "orig in F_C0": [i in F_C0 for i in rows0.tolist()],    # F_C0 = rho >= tau candidates of the original fit (no judge)
    "transformed in F_C": [j in F_C for j in cols1.tolist()],
}).sort_values("Jaccard", ascending=False, ignore_index=True)

mj = match_df["Jaccard"]
print("matched Jaccard quantiles: " + "  ".join(f"q{int(q * 100)}={mj.quantile(q):.3f}" for q in (0, .1, .25, .5, .75, .9, 1)))
print(f"pairs with Jaccard >= 0.5: {(mj >= 0.5).sum()} / {RANK}   >= 0.8: {(mj >= 0.8).sum()} / {RANK}   "
      f"matched partner = best partner: {(match_df['transformed feature'] == match_df['best partner']).sum()} / {RANK}")
print(f"|cos| of mapped directions: median={match_df['|cos(z0, unmap(z~))|'].median():.3f}   "
      f"Spearman vs Jaccard = {match_df['Jaccard'].corr(match_df['|cos(z0, unmap(z~))|'], method='spearman'):.3f}")
print("\nhistogram of matched Jaccard:")
print(pd.cut(mj, bins=np.linspace(0, 1, 11), include_lowest=True).value_counts().sort_index().to_string())
with pd.option_context("display.max_rows", None):
    display(match_df)

|S0_i| (original widths): min=10 median=15 max=239   |S_j| (transformed widths): min=10 median=30 max=239
matched Jaccard quantiles: q0=0.000  q10=0.000  q25=0.012  q50=0.178  q75=0.455  q90=0.650  q100=0.917
pairs with Jaccard >= 0.5: 24 / 100   >= 0.8: 5 / 100   matched partner = best partner: 59 / 100
|cos| of mapped directions: median=0.374   Spearman vs Jaccard = 0.815

histogram of matched Jaccard:
Jaccard
(-0.001, 0.1]    44
(0.1, 0.2]        7
(0.2, 0.3]        9
(0.3, 0.4]        7
(0.4, 0.5]        9
(0.5, 0.6]        8
(0.6, 0.7]        9
(0.7, 0.8]        2
(0.8, 0.9]        3
(0.9, 1.0]        2


,orig feature,transformed feature,|S0|,|S|,|S0∩S|,Jaccard,best Jaccard,best partner,"|cos(z0, unmap(z~))|",orig in F_C0,transformed in F_C
0,12,65,11,12,11,0.916667,0.916667,65,0.908036,True,False
1,86,39,10,11,10,0.909091,0.909091,39,0.956983,True,False
2,24,83,22,23,21,0.875000,0.875000,83,0.836027,True,False
3,35,36,10,10,9,0.818182,0.818182,36,0.895652,True,False
4,67,93,106,117,100,0.813008,0.813008,93,0.478245,True,False
5,42,42,15,17,14,0.777778,0.777778,42,0.786243,False,False
6,51,34,18,25,18,0.720000,0.720000,34,0.746808,True,False
7,52,26,48,54,42,0.700000,0.700000,26,0.593071,False,False
8,56,56,213,239,185,0.692884,0.692884,56,0.524015,False,False
9,18,22,17,25,17,0.680000,0.680000,22,0.837642,True,False


#### Concept features in both fits

For each transformed concept feature $f\in F_C$ and each original $\rho\ge\tau$ candidate $f_0\in F_{C0}$ (no judge on the original fit, see 3.2.e′): its matched partner in
the other fit, their Jaccard, and whether the partner is itself a concept feature of its own fit. The token table splits
the union of the two supports into *both* / *only original* / *only transformed*, with labels (C / N / B) and the signed
$Y^0$ and $Y$ values, sorted by the larger magnitude within each group.

In [42]:
def pair_table(i: int, j: int) -> pd.DataFrame:
    """Union of S0_i and S_j: membership, label, Y0[i, t], Y[j, t]; grouped, then sorted by magnitude."""
    cols = np.where(S0[i] | S1[j])[0]
    where = np.where(S0[i, cols] & S1[j, cols], "both", np.where(S0[i, cols], "only original", "only transformed"))
    df = pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens([vstar_ids[t] for t in cols]),
        "label": [LABEL_TAG[codes[t]] for t in cols],
        "in": where,
        f"Y0[{i}]": Y0[i, cols].numpy(),
        f"Y[{j}]": Y[j, cols].numpy(),
    })
    group = df["in"].map({"both": 0, "only original": 1, "only transformed": 2})
    mag = np.maximum(df[f"Y0[{i}]"].abs(), df[f"Y[{j}]"].abs())
    return df.assign(_g=group, _m=mag).sort_values(["_g", "_m"], ascending=[True, False]) \
             .drop(columns=["_g", "_m"]).reset_index(drop=True)


pairs = {}
for f in F_C:
    i = match1[f]
    pairs.setdefault((i, f), []).append(
        f"transformed concept feature {f}: matched original feature {i} is {'IN' if i in F_C0 else 'NOT in'} F_C0 "
        f"(best-Jaccard original partner: {int(J[:, f].argmax())}, J={J[:, f].max():.3f})")
for f0 in F_C0:
    j = match0[f0]
    pairs.setdefault((f0, j), []).append(
        f"original rho-candidate {f0}: matched transformed feature {j} is {'IN' if j in F_C else 'NOT in'} F_C "
        f"(best-Jaccard transformed partner: {int(J[f0].argmax())}, J={J[f0].max():.3f})")

print(f"F_C (transformed, judged) = {F_C}   F_C0 (original, rho >= tau candidates, not judged) = {F_C0}   "
      f"matching maps F_C0 onto F_C exactly: {sorted(match0[f0] for f0 in F_C0) == F_C}\n")
if not pairs:
    print("F_C and F_C0 are both empty: nothing to compare.")

for (i, j), notes in sorted(pairs.items()):
    tab = pair_table(i, j)
    n_in = tab["in"].value_counts()
    cos_ij = F.cosine_similarity(Z0[:, i], Z_back[:, j], dim=0).abs().item()
    print(f"{'=' * 100}\n=== original {i} <-> transformed {j}   Jaccard = {J[i, j]:.3f}   "
          f"|cos(z0, unmap(z~))| = {cos_ij:.3f}\n{'=' * 100}")
    for note in notes:
        print("  " + note)
    print(f"  both: {n_in.get('both', 0)}   only original: {n_in.get('only original', 0)}   "
          f"only transformed: {n_in.get('only transformed', 0)}")
    display(pd.crosstab(tab["in"], tab["label"]))
    with pd.option_context("display.max_rows", None):
        display(tab)

F_C (transformed, judged) = [46]   F_C0 (original, rho >= tau candidates, not judged) = [3, 5, 6, 10, 12, 18, 20, 22, 23, 24, 27, 28, 29, 30, 32, 35, 36, 39, 40, 41, 43, 50, 51, 54, 57, 58, 59, 61, 67, 70, 71, 73, 76, 78, 80, 81, 82, 83, 84, 85, 86, 87, 89, 91, 93, 94, 96, 98]   matching maps F_C0 onto F_C exactly: False

=== original 3 <-> transformed 44   Jaccard = 0.077   |cos(z0, unmap(z~))| = 0.019
  original rho-candidate 3: matched transformed feature 44 is NOT in F_C (best-Jaccard transformed partner: 30, J=0.288)
  both: 2   only original: 14   only transformed: 10


label,B,C,N
in,,,
both,0,2,0
only original,0,10,4
only transformed,1,8,1


,token,label,in,Y0[3],Y[44]
0,▁learn,C,both,-0.228208,0.104365
1,▁learns,C,both,-0.204199,0.079621
2,▁teaching,C,only original,-0.426817,0.000000
3,▁teach,C,only original,-0.425554,0.000000
4,▁teaches,C,only original,-0.381216,0.000000
5,▁teachings,C,only original,-0.316115,-0.000000
6,▁lessons,C,only original,-0.233969,0.000000
7,▁teacher,C,only original,-0.229926,-0.000000
8,▁education,N,only original,-0.203088,0.000000
9,▁educational,N,only original,-0.202053,0.000000


=== original 5 <-> transformed 80   Jaccard = 0.022   |cos(z0, unmap(z~))| = 0.349
  original rho-candidate 5: matched transformed feature 80 is NOT in F_C (best-Jaccard transformed partner: 80, J=0.022)
  both: 1   only original: 28   only transformed: 16


label,B,C,N
in,,,
both,0,1,0
only original,0,18,10
only transformed,2,5,9


...

label,B,C,N
in,,,
both,1,3,0
only original,1,3,2
only transformed,1,4,2


,token,label,in,Y0[96],Y[60]
0,▁half,C,both,0.265123,-0.543030
1,half,B,both,0.257016,-0.516784
2,▁Half,C,both,0.250305,-0.503380
3,▁partial,C,both,0.188457,-0.193031
4,part,C,only original,0.424565,-0.000000
5,▁part,B,only original,0.417570,-0.000000
6,▁Part,N,only original,0.399360,-0.000000
7,▁parts,C,only original,0.356414,0.000000
8,▁Parts,C,only original,0.337679,0.000000
9,▁components,N,only original,0.101572,0.000000


=== original 98 <-> transformed 19   Jaccard = 0.227   |cos(z0, unmap(z~))| = 0.741
  original rho-candidate 98: matched transformed feature 19 is NOT in F_C (best-Jaccard transformed partner: 19, J=0.227)
  both: 10   only original: 0   only transformed: 34


label,B,C,N
in,,,
both,0,9,1
only transformed,5,23,6


,token,label,in,Y0[98],Y[19]
0,▁love,C,both,0.484838,-0.356023
1,▁Love,N,both,0.443047,-0.276587
2,Love,C,both,0.433316,-0.272849
3,▁loves,C,both,0.399589,-0.317589
4,▁loved,C,both,0.364449,-0.269047
5,▁likes,C,both,0.181497,-0.295180
6,▁dislike,C,both,0.143854,-0.144932
7,▁favourite,C,both,0.120298,-0.086338
8,▁romance,C,both,0.102466,-0.092846
9,▁joy,C,both,0.098889,-0.071809


#### Run summary → `RUN_DIR/summary.json`

Configuration (variant, `VWConfig`, whitening), $|V^*|$, relative errors and width statistics of both fits, $\rho_{\max}$, $F_C$
(judged, whitened fit) with each feature's width, `F_C0` (= $\rho\ge\tau$ candidates of the original fit, not judged), each $F_C$
feature's matched original partner (Jaccard, partner in `F_C0`?), and the number of concept-only tokens $F_C$ touches.

In [43]:
def write_summary(**extra) -> dict:
    """Write RUN_DIR/summary.json from the current state; `extra` adds or overrides fields."""
    eligible_cols = ((Y[F_C].abs() > 0).any(dim=0).numpy() & (codes == 1)) if F_C else np.zeros(len(vstar_ids), bool)
    summary = {
        "config": {"model": MODEL_NAME, "concept": CONCEPT, "variant": VARIANT, "vwconfig": CFG.to_dict(), "seed": SEED,
                   "background": BACKGROUND, "target": TARGET, "gamma": GAMMA, "eta": ETA, "eps_rel": EPS_REL,
                   "eps": float(bw.eps), "rank": RANK, "smoke": SMOKE, "n_target": len(target_ids)},
        "n_vstar": len(vstar_ids),
        "rel_err": {"original": rel_err0, "transformed": rel_err},
        "n_iter": {"original": fit0["n_iter"], "transformed": fit["n_iter"]},
        "widths": {"transformed": widths.tolist(), "original": widths0.tolist()},
        "F_C_widths": {str(f): int(widths[f]) for f in F_C},
        "rho_max": float(stats["ratio_abs"].max()),
        "rho_max_original": float(stats0["ratio_abs"].max()),
        "n_F_C": len(F_C),
        "F_C": F_C,
        "F_C0": F_C0,                     # rho >= tau candidates of the original fit (no judge)
        "F_C_matched": [{"feature": f, "partner": match1[f], "jaccard": float(J[match1[f], f]),
                         "partner_in_F_C0": match1[f] in F_C0, "cos_m": float(cos_m[f])} for f in F_C],
        "F_C0_matched": [{"feature": f0, "partner": match0[f0], "jaccard": float(J[f0, match0[f0]]),
                          "partner_in_F_C": match0[f0] in F_C} for f0 in F_C0],
        "matched_jaccard_median": float(np.median(J[rows0, cols1])),
        "n_edit": int(eligible_cols.sum()),
        **extra,
    }
    save_json(RUN_DIR / "summary.json", summary)
    print(f"wrote {RUN_DIR / 'summary.json'}: F_C={F_C} F_C0={F_C0} n_edit={summary['n_edit']} "
          f"rel_err={rel_err:.4f} (orig {rel_err0:.4f}) rho_max={summary['rho_max']:.2f}")
    return summary


summary = write_summary()

wrote runs/variable_width/Harry_Potter/m1/summary.json: F_C=[46] F_C0=[3, 5, 6, 10, 12, 18, 20, 22, 23, 24, 27, 28, 29, 30, 32, 35, 36, 39, 40, 41, 43, 50, 51, 54, 57, 58, 59, 61, 67, 70, 71, 73, 76, 78, 80, 81, 82, 83, 84, 85, 86, 87, 89, 91, 93, 94, 96, 98] n_edit=44 rel_err=0.8510 (orig 0.8984) rho_max=129.35
